# VisualOddball EEG Preprocessing

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## Analyze the preprocessing

In [ ]:
pip install mne

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.5/7.5 MB 36.5 MB/s eta 0:00:00


## Matlab Preperation Steps

NOTE: First, create 1 folder `content/data/` in your colab environment. Then upload all 6 files of all tasks for all subjects you want to preprocess in the `content/data/` folder.

In [ ]:
!wget -q https://www.mathworks.com/mpm/glnxa64/mpm -O mpm
!chmod +x mpm
!./mpm install --release=R2025a --products=MATLAB
!apt-get update -y
!apt-get install -y \
  libxcomposite1 libatk1.0-0 libgtk-3-0 libasound2 libxrandr2 libxtst6 libnss3 \
  libxrender1 libxfixes3 libcairo2 libpangocairo-1.0-0 libpango-1.0-0 \
  libgdk-pixbuf-2.0-0 libxcb1 libxi6 build-essential python3-dev

Installing with the following parameters:
--release=R2025a
--products=MATLAB
---------------------------------------------
The following MathWorks Products are licensed under the The MathWorks, Inc. Software License
Agreement, available in the installation of the MathWorks Product or in the virtual machine image.
MATLAB

Starting install
Products will be installed to: /usr/local/MATLAB/R2025a
Finished install
Completing setup...
Installation complete
Hit:1 https://cli.github.com/packages stable InRelease
Get:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:4 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:5 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Hit:6 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:7 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:8 https://developer.download.

In [ ]:
!/usr/local/MATLAB/R2025a/bin/matlab -nodesktop -nosplash -licmode onlinelicensing

=
                            < M A T L A B (R) >
                  Copyright 1984-2024 The MathWorks, Inc.
             R2025a Update 1 (25.1.0.2973910) 64-bit (glnxa64)
                                July 3, 2025

 
To get started, type doc.
For product information, visit www.mathworks.com.
 
>> quit
>

In [ ]:
# Point Python to MATLAB, then install the MATLAB Engine from the MATLAB install
import os
os.environ["MATLABROOT"] = "/usr/local/MATLAB/R2025a"
os.environ["LD_LIBRARY_PATH"] = os.environ.get("LD_LIBRARY_PATH","") + ":/usr/local/MATLAB/R2025a/bin/glnxa64"

!python3 -m pip -q install -U pip wheel setuptools
!python3 -m pip -q install $MATLABROOT/extern/engines/python

# Start MATLAB Engine using online licensing
import matlab.engine
eng = matlab.engine.start_matlab("-licmode onlinelicensing")
print("MATLAB started via Engine")

# Install EEGLAB, ERPLAB, and BrainVision plugin; add everything to path
!rm -rf /content/eeglab /content/erplab
!wget -q https://github.com/sccn/eeglab/archive/refs/heads/develop.zip -O /content/eeglab.zip
!unzip -q -o /content/eeglab.zip -d /content && rm -f /content/eeglab.zip
!mv -f /content/eeglab-develop /content/eeglab

!wget -q https://github.com/lucklab/erplab/archive/refs/heads/master.zip -O /content/erplab.zip
!unzip -q -o /content/erplab.zip -d /content && rm -f /content/erplab.zip
!mv -f /content/erplab-master /content/erplab

# BrainVision I/O plugin (pop_loadbv)
!rm -rf /content/eeglab/plugins/bva-io
!wget -q https://github.com/sccn/bva-io/archive/refs/heads/master.zip -O /content/bva-io.zip
!unzip -q -o /content/bva-io.zip -d /content && rm -f /content/bva-io.zip
!mv /content/bva-io-master /content/eeglab/plugins/bva-io

# Add all subfolders to MATLAB path in one shot
p = eng.genpath('/content/eeglab')
eng.addpath(p, nargout=0)
p = eng.genpath('/content/erplab')
eng.addpath(p, nargout=0)

# Define root paths inside Drive-mounted DataLiteracyProject
DATA_ROOT = "/content/drive/MyDrive/DataLiteracyProject"
PROJECT_ROOT = f"{DATA_ROOT}/Preprocessed_VisualOddball"

# Create project folders under DataLiteracyProject/Preprocessed_VisualOddball
!mkdir -p "$PROJECT_ROOT/01_raw" "$PROJECT_ROOT/02_preprocessed" "$PROJECT_ROOT/03_preICA"

print("eeglab.m:", eng.which('eeglab'))
print("pop_loadbv:", eng.which('pop_loadbv'))
print("pop_erplabDeleteTimeSegments:", eng.which('pop_erplabDeleteTimeSegments'))
print("Setup complete.")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 23.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 42.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 7.34.0 requires jedi>=0.16, which is not installed.
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
MATLAB started via Engine
eeglab.m: /content/eeglab/eeglab.m
pop_loadbv: /content/eeglab/plugins/bva-io/pop_loadbv.m
pop_erplabDeleteTimeSegments: /content/erplab/pop_functions/pop_erplabDeleteTimeSegments.m
Setup complete.


In [ ]:
# install dipfit under the EEGLAB plugins folder
!rm -rf /content/eeglab/plugins/dipfit
!wget -q https://github.com/sccn/dipfit/archive/refs/heads/master.zip -O dipfit.zip
!unzip -q -o dipfit.zip
!rm -f dipfit.zip
!mv dipfit-master /content/eeglab/plugins/dipfit

# install the firfilt plugin
!rm -rf /content/eeglab/plugins/firfilt
!wget -q https://github.com/sccn/firfilt/archive/refs/heads/master.zip -O firfilt.zip
!unzip -q -o firfilt.zip
!rm -f firfilt.zip
!mv firfilt-master /content/eeglab/plugins/firfilt

In [ ]:
# install ICLabel
!wget -q https://github.com/sccn/ICLabel/archive/refs/heads/master.zip -O /content/ICLabel.zip
!unzip -q -o /content/ICLabel.zip -d /content
!rm -f /content/ICLabel.zip
!mv /content/ICLabel-master /content/eeglab/plugins/ICLabel


In [ ]:
# install Viewprops plugin
!rm -rf /content/eeglab/plugins/viewprops
!wget -q https://github.com/sccn/viewprops/archive/refs/heads/master.zip -O /content/viewprops.zip
!unzip -q -o /content/viewprops.zip -d /content
!rm -f /content/viewprops.zip
!mv /content/viewprops-master /content/eeglab/plugins/viewprops

# refresh MATLAB path and verify
p = eng.genpath('/content/eeglab'); eng.addpath(p, nargout=0)
print("pop_viewprops:", eng.which('pop_viewprops'))


pop_viewprops: /content/eeglab/plugins/viewprops/pop_viewprops.m


In [ ]:
# ICLabel + MatConvNet CPU-only setup that works with R2025a on Colab

# Make sure Parallel Computing Toolbox is available so vl_compilenn can find toolboxdir('parallel')
!wget -q https://www.mathworks.com/mpm/glnxa64/mpm -O mpm
!chmod +x mpm
!./mpm install --release=R2025a --products=Parallel_Computing_Toolbox

# Install ICLabel and Viewprops with submodules
!rm -rf /content/eeglab/plugins/ICLabel /content/eeglab/plugins/viewprops
!git clone --depth 1 --recursive https://github.com/sccn/ICLabel.git /content/eeglab/plugins/ICLabel
!git clone --depth 1 https://github.com/sccn/viewprops.git /content/eeglab/plugins/viewprops

# Keep exactly one MatConvNet under ICLabel/dependencies/matconvnet
import os, shutil
dep_mcn = "/content/eeglab/plugins/ICLabel/dependencies/matconvnet"
top_mcn = "/content/eeglab/plugins/ICLabel/matconvnet"
if not os.path.exists(dep_mcn):
    os.makedirs("/content/eeglab/plugins/ICLabel/dependencies", exist_ok=True)
    if os.path.exists(top_mcn):
        shutil.move(top_mcn, dep_mcn)
else:
    if os.path.exists(top_mcn):
        shutil.rmtree(top_mcn)

# Patch CUDA sources so CPU parses are OK
!sed -i '1i #include <limits>\n#include <cmath>' /content/eeglab/plugins/ICLabel/dependencies/matconvnet/matlab/src/bits/nnpooling.cu || true
!sed -i '1i #include <limits>\n#include <cmath>' /content/eeglab/plugins/ICLabel/dependencies/matconvnet/matlab/src/bits/nnnormalize.cu || true

# Build tools
!apt-get update -qq
!apt-get install -y -qq build-essential g++

# Start MATLAB engine and refresh paths
import matlab.engine, os
os.environ["LD_LIBRARY_PATH"] = os.environ.get("LD_LIBRARY_PATH","") + ":/usr/local/MATLAB/R2025a/bin/glnxa64"
try:
    eng
except NameError:
    eng = matlab.engine.start_matlab("-licmode onlinelicensing")
p = eng.genpath('/content/eeglab'); eng.addpath(p, nargout=0)
eng.addpath('/content', nargout=0)
eng.eeglab('nogui', nargout=0)

# Make MATLAB see newly installed toolboxes
eng.eval("rehash toolboxcache", nargout=0)

# Configure MEX compilers
eng.eval("mex -setup C", nargout=0)
eng.eval("mex -setup C++", nargout=0)

# Compile MatConvNet CPU-only and activate
eng.eval("addpath('/content/eeglab/plugins/ICLabel/dependencies/matconvnet/matlab');", nargout=0)
eng.eval("setenv('MEX_CXXFLAGS', [getenv('MEX_CXXFLAGS') ' -std=c++11']);", nargout=0)
eng.eval("vl_compilenn('enableGpu', false, 'enableImreadJpeg', false, 'verbose', 2);", nargout=0)
eng.eval("vl_setupnn;", nargout=0)

# Compatibility shims some ICLabel builds expect

# dagnn_bc.DagNN.loadobj -> dagnn.DagNN.loadobj
os.makedirs('/content/eeglab/plugins/ICLabel/+dagnn_bc', exist_ok=True)
shim_path = '/content/eeglab/plugins/ICLabel/+dagnn_bc/DagNN.m'
if not os.path.exists(shim_path):
    with open(shim_path, 'w') as f:
        f.write("""classdef DagNN < dagnn.DagNN
  methods (Static)
    function obj = loadobj(s)
      obj = dagnn.DagNN.loadobj(s);
    end
  end
end
""")

# Minimal dagnn.Reshape layer if missing
reshape_dir = '/content/eeglab/plugins/ICLabel/dependencies/matconvnet/matlab/+dagnn/@Reshape'
os.makedirs(reshape_dir, exist_ok=True)
reshape_cls = os.path.join(reshape_dir, 'Reshape.m')
if not os.path.exists(reshape_cls):
    with open(reshape_cls, 'w') as f:
        f.write("""classdef Reshape < dagnn.Layer
  properties
    shape = []
  end
  methods
    function outputs = forward(obj, inputs, params)
      x = inputs{1};
      tgt = obj.shape;
      if isempty(tgt)
        tgt = size(x);
      else
        if any(tgt == -1)
          known = prod(tgt(tgt~=-1));
          tgt(tgt == -1) = numel(x) / known;
        end
      end
      outputs{1} = reshape(x, tgt);
    end
    function [derInputs, derParams] = backward(obj, inputs, params, derOutputs)
      derInputs{1} = reshape(derOutputs{1}, size(inputs{1}));
      derParams = {};
    end
    function obj = Reshape(varargin)
      obj.load(varargin{:});
    end
  end
end
""")

# Refresh paths and sanity check
p = eng.genpath('/content/eeglab'); eng.addpath(p, nargout=0)
eng.eeglab('nogui', nargout=0)
print("iclabel:", eng.which('iclabel'))
print("pop_viewprops:", eng.which('pop_viewprops'))
print("vl_setupnn:", eng.which('vl_setupnn'))


Installing with the following parameters:
--release=R2025a
--products=Parallel_Computing_Toolbox
---------------------------------------------
The following MathWorks Products are licensed under the The MathWorks, Inc. Software License
Agreement, available in the installation of the MathWorks Product or in the virtual machine image.
Parallel_Computing_Toolbox

Starting install
Products will be installed to: /usr/local/MATLAB/R2025a
Finished install
Completing setup...
Installation complete
Cloning into '/content/eeglab/plugins/ICLabel'...
remote: Enumerating objects: 30, done.
remote: Counting objects: 100% (30/30), done.
remote: Compressing objects: 100% (28/28), done.
remote: Total 30 (delta 1), reused 10 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (30/30), 33.68 MiB | 40.43 MiB/s, done.
Resolving deltas: 100% (1/1), done.
Submodule 'matconvnet' (https://github.com/lucapton/matconvnet.git) registered for path 'matconvnet'
Submodule 'viewprops' (https://github.com/lucapt

## script 1

The subjects' corresponding `.eeg`, `.vhdr`, and `.vmrk` files should now be uploaded to a folder named "data" in the colab `content/` path.

In [ ]:
%%writefile /content/IDEA_P3b_01_a_importbv.m
function IDEA_P3b_01_a_importbv
% Imports BrainVision .vhdr files from
%   /content/data/DataLiteracyProject/OriginalData/sub-XXX/eeg
% and writes EEGLAB .set
% Output goes to
%   /content/data/DataLiteracyProject/Preprocessed_VisualOddball/01_raw

close all; clearvars;

% start EEGLAB without GUI
eeglab nogui;

data_root = '/content/drive/MyDrive/DataLiteracyProject';
bvroot    = [data_root '/OriginalData'];
project   = [data_root '/Preprocessed_VisualOddball/'];
newpath   = [project '01_raw/'];

if ~exist(newpath,'dir'); mkdir(newpath); end

% recursively find visualoddball .vhdr under sub-*/eeg
folderinfo = dir(fullfile(bvroot, 'sub-*', 'eeg', '*visualoddball*.vhdr'));
if isempty(folderinfo)
    error('No visualoddball .vhdr files found under %s/sub-*/eeg', bvroot);
end

subject_list = folderinfo;
numsubject   = numel(subject_list);


for s = 1:numsubject
    subject  = subject_list(s).name;   % e.g. "sub-001_task-visualoddball_eeg.vhdr"
    subjpath = subject_list(s).folder; % e.g. ".../OriginalData/sub-001/eeg"
    fprintf('Loading %s\n', fullfile(subjpath, subject));

    % Load BV data
    EEG = pop_loadbv(subjpath, subject);

    % Save as .set
    ID      = extractBefore(subject,'.');  % "sub-001_task-visualoddball_eeg"
    setname = [ID '_raw'];
    EEG     = pop_editset(EEG, 'setname', setname);
    eegfilename = [setname '.set'];
    pop_saveset(EEG, 'filename', eegfilename, 'filepath', newpath);

    eeglab redraw;
end

fprintf('Done. Wrote %d file(s) to %s\n', numsubject, newpath);
end


Overwriting /content/IDEA_P3b_01_a_importbv.m


In [ ]:
import matlab.engine, os
os.environ["LD_LIBRARY_PATH"] = os.environ.get("LD_LIBRARY_PATH","") + ":/usr/local/MATLAB/R2025a/bin/glnxa64"
try:
    eng
except NameError:
    eng = matlab.engine.start_matlab("-licmode onlinelicensing")
eng.addpath('/content', nargout=0)
print("which:", eng.which('IDEA_P3b_01_a_importbv'))
eng.IDEA_P3b_01_a_importbv(nargout=0)
!ls -1 "/content/drive/MyDrive/DataLiteracyProject/Preprocessed_VisualOddball/01_raw"

which: /content/IDEA_P3b_01_a_importbv.m


## script 2

The dataset uses a 32-channel actiCAP arranged in the standard 10/20 layout, ground at FPz, online reference at Cz, and two electrodes repurposed as mastoids for offline rereferencing. They do not provide digitized 3D positions per subject, so a standard montage was considered appropriate.

In [ ]:
%%writefile IDEA_P3b_02_preprocess_script.m
function IDEA_P3b_02_preprocess_script
clearvars;
eeglab nogui;

% Folders
data_root = '/content/drive/MyDrive/DataLiteracyProject';
project   = [data_root '/Preprocessed_VisualOddball/'];
rawfolder = [project '01_raw/'];
outfolder = [project '02_preprocessed/'];
if ~exist(outfolder, 'dir'); mkdir(outfolder); end

% Find input files
cd(rawfolder);
files = dir('*raw.set');
if isempty(files)
    error('No *raw.set files found in %s. Run the import step first.', rawfolder);
end

% Process all files found
for k = 1:numel(files)
    inname = files(k).name;
    ID = extractBefore(inname, 'raw.set');
    fprintf('\n=== Preprocessing %s ===\n', inname);
    % Load
    EEG = pop_loadset(inname);
    EEG = eeg_checkset(EEG);
    chanloc_tsv = [data_root '/IDEA_BV_chanlocs.tsv'];
    if exist(chanloc_tsv, 'file')
        fprintf('Loading channel locations from TSV: %s\n', chanloc_tsv);
        EEG = pop_chanedit(EEG, 'load', {chanloc_tsv, 'filetype', 'tsv'});
    else
        stdelc = '/content/eeglab/plugins/dipfit/standard_BEM/elec/standard_1005.elc';
        if ~exist(stdelc, 'file')
            error('standard_1005.elc not found at %s. Make sure EEGLAB is installed.', stdelc);
        end
        fprintf('TSV not found. Using built-in 10-05 lookup: %s\n', stdelc);
        EEG = pop_chanedit(EEG, 'lookup', stdelc);
    end
    EEG = eeg_checkset(EEG);

    % Rereference to mastoids by LABEL
    % Accept common label variants
    labs = lower(string({EEG.chanlocs.labels}));
    left_candidates  = ismember(labs, ["lm","m1","tp9","ma1","mastoidl","mastoid-left","tp9h"]);
    right_candidates = ismember(labs, ["rm","m2","tp10","ma2","mastoidr","mastoid-right","tp10h"]);
    li = find(left_candidates, 1);
    ri = find(right_candidates, 1);
    if isempty(li) || isempty(ri)
        error('Could not find mastoid channels by label. Available labels include: %s', strjoin(labs(1:min(end,40)), ', '));
    end
    fprintf('Rereferencing to mastoids: L=%s (%d) R=%s (%d)\n', EEG.chanlocs(li).labels, li, EEG.chanlocs(ri).labels, ri);
    EEG = pop_reref(EEG, [li ri], 'keepref','on');
    EEG = eeg_checkset(EEG);

    % Create bipolar EOG channels by LABEL if possible
    % VEOGR = VER - FP2, HEOG = HER - HEL
    % If any label is missing, skip with a warning
    lab2idx = containers.Map(lower(string({EEG.chanlocs.labels})), num2cell(1:numel(EEG.chanlocs)));
    have = @(s) isKey(lab2idx, lower(string(s)));
    idx = @(s) lab2idx(lower(string(s)));

    try
        if have('VER') && have('FP2')
            expr1 = sprintf('ch32 = ch%d - ch%d label VEOGR', idx('VER'), idx('FP2'));
        else
            warning('Skipping VEOGR. Need VER and FP2 labels.');
            expr1 = '';
        end
        if have('HER') && have('HEL')
            expr2 = sprintf('ch33 = ch%d - ch%d label HEOG', idx('HER'), idx('HEL'));
        else
            warning('Skipping HEOG. Need HER and HEL labels.');
            expr2 = '';
        end
        ops = {};
        if ~isempty(expr1), ops{end+1} = expr1; end %#ok<AGROW>
        if ~isempty(expr2), ops{end+1} = expr2; end %#ok<AGROW>
        if ~isempty(ops)
            EEG = pop_eegchanoperator(EEG, ops);
            EEG = eeg_checkset(EEG);
        end
    catch ME
        warning('EOG derivation skipped due to: %s', ME.message);
    end

    % Bandpass filter 0.1 to 40 Hz
    EEG = pop_eegfiltnew(EEG, 'locutoff', 0.1, 'hicutoff', 40);
    EEG = eeg_checkset(EEG);

    % Save
    EEG.setname = [ID 'preprocessed'];
    outname = [EEG.setname '.set'];
    pop_saveset(EEG, 'filename', outname, 'filepath', outfolder);
    fprintf('Saved %s\n', fullfile(outfolder, outname));
end

fprintf('\nAll done. Outputs in %s\n', outfolder);
end


Overwriting IDEA_P3b_02_preprocess_script.m


In [ ]:
eng.IDEA_P3b_02_preprocess_script(nargout=0)

## script 3

In [ ]:
%%writefile IDEA_P3b_03_1_badchandetectpreICA.m
function IDEA_P3b_03_1_badchandetectpreICA
close all; clearvars;
eeglab nogui;

% make sure ERPLAB is on the path
if exist('pop_erplabDeleteTimeSegments','file') ~= 2
    addpath(genpath('/content/erplab'));
end

data_root = '/content/drive/MyDrive/DataLiteracyProject';
project   = [data_root '/Preprocessed_VisualOddball/'];
main      = [project '02_preprocessed/'];
preICA    = [project '03_preICA/'];
if ~exist(preICA,'dir'); mkdir(preICA); end

% list preprocessed sets
cd(main);
folderinfo = dir('*.set');
if isempty(folderinfo)
    error('No .set files found in %s. Run script 2 first.', main);
end

% try lab Excel, else we will build from EEG on first file
filesheet = [data_root '/IDEA_BV_channellocations.xlsx'];
use_excel = exist(filesheet,'file') == 2;
if use_excel
    try
        base_chans = readtable(filesheet, 'Sheet', 'preICAbadchandetect', 'NumHeaderLines',0);
        % expect columns Number and Label at least
        if ~all(ismember({'Number','Label'}, base_chans.Properties.VariableNames))
            error('Excel sheet must contain columns Number and Label');
        end
    catch ME
        warning('Could not read Excel sheet: %s. Will build channels from EEG.', ME.message);
        use_excel = false;
    end
end

subject_list = {folderinfo.name}.';
numsubject   = numel(subject_list);

% will preallocate after we know channel count
loss = [];
preICAchans = [];
ID = cell(numsubject,1);

for s = 1:numsubject
    subject = subject_list{s};
    ID{s} = extractBefore(subject,'_preprocessed.set');

    % load EEG
    EEG = pop_loadset('filename', subject);

    % if no Excel, build channel table from this EEG
    if s == 1
        if ~use_excel
            nCh = size(EEG.data,1);
            labels = string({EEG.chanlocs.labels}.');
            if numel(labels) ~= nCh
                labels = string(compose('Ch%d', 1:nCh)).';
            end
            base_chans = table((1:nCh).', labels, 'VariableNames', {'Number','Label'});
        end
        nCh = height(base_chans);
        loss = zeros(numsubject, nCh);
        preICAchans = nan(numsubject, nCh);
    end

    % remove long no event segments
    EEG = pop_erplabDeleteTimeSegments(EEG, 'displayEEG', 0, ...
        'startEventcodeBufferMS', 3000, 'endEventcodeBufferMS', 3000, ...
        'timeThresholdMS', 6000);
    A = EEG;

    % per channel extreme artifact detection
    chans_s = base_chans;
    chans_s.loss = zeros(nCh,1);
    for j = 1:nCh
        chan = chans_s.Number(j);
        B = pop_continuousartdet(A, 'ampth', 300, 'winms', 500, ...
            'stepms', 50, 'chanArray', chan, 'review', 'off');
        chans_s.loss(j) = (A.pnts - B.pnts) / A.pnts * 100;
    end

    % z scores and include list
    z = local_zscore(chans_s.loss);
    include = nan(nCh,1);
    for j = 1:nCh
        if chans_s.loss(j) <= 10 && z(j) < 3.29
            include(j) = chans_s.Number(j);
        end
    end

    loss(s,:) = chans_s.loss.';
    preICAchans(s,:) = include.';
end

% write loss table
loss_tbl = array2table(loss);
% column names from labels if available
if ismember('Label', base_chans.Properties.VariableNames)
    loss_tbl.Properties.VariableNames = matlab.lang.makeValidName(base_chans.Label);
end
loss_tbl = addvars(loss_tbl, ID, 'Before', 1, 'NewVariableNames', 'ID');
projectname = 'COCOA';
writetable(loss_tbl, [preICA projectname '_preICAextremeloss.xlsx'], ...
    'WriteMode','append', 'WriteVariableNames', true, 'WriteRowNames', true);

% write channel include table
pre_tbl = array2table(preICAchans);
pre_tbl = addvars(pre_tbl, ID, 'Before', 1, 'NewVariableNames', 'ID');
writetable(pre_tbl, [preICA projectname '_preICAchannels.xlsx'], ...
    'WriteMode','append', 'WriteVariableNames', true, 'WriteRowNames', true);

fprintf('Done. Wrote Excel outputs to %s\n', preICA);
end

function z = local_zscore(x)
% Minimal z-score without the Statistics Toolbox; NaN-safe
x = double(x);
mu = mean(x, 'omitnan');
sd = std(x, 0, 'omitnan');   % population flag 0 like zscore default
if isnan(sd) || sd == 0
    z = zeros(size(x));      % avoid divide-by-zero
else
    z = (x - mu) ./ sd;
end
end


Overwriting IDEA_P3b_03_1_badchandetectpreICA.m


In [ ]:
eng.IDEA_P3b_03_1_badchandetectpreICA(nargout=0)

## script 4

In [ ]:
%%writefile IDEA_P3b_03_2_preICA.m
function IDEA_P3b_03_2_preICA
% Pre ICA prep:
% load *_preprocessed.set from 02_preprocessed
% remove long no-event segments, run extreme artifact detector on chosen chans
% save *_preICA.set to 03_preICA

close all; clearvars;
eeglab nogui;

% make sure ERPLAB is on path
if exist('pop_erplabDeleteTimeSegments','file') ~= 2
    addpath(genpath('/content/erplab'));
end

data_root = '/content/drive/MyDrive/DataLiteracyProject';
project   = [data_root '/Preprocessed_VisualOddball/'];
inDir     = [project '02_preprocessed/'];
outDir    = [project '03_preICA/'];
if ~exist(outDir,'dir'); mkdir(outDir); end

% optional spreadsheet produced by the previous step
xlsPath  = [outDir 'COCOA_preICAchannels.xlsx'];
haveXLS  = exist(xlsPath,'file') == 2;

if haveXLS
    try
        T = readtable(xlsPath);
        % first column should be ID
        if ~ismember('ID', T.Properties.VariableNames)
            warning('preICAchannels sheet has no ID column. Ignoring spreadsheet and using data driven selection.');
            haveXLS = false;
        end
    catch ME
        warning('Could not read %s (%s). Proceeding without spreadsheet.', xlsPath, ME.message);
        haveXLS = false;
    end
end

cd(inDir);
files = dir('*_preprocessed.set');
if isempty(files), error('No *_preprocessed.set in %s', inDir); end

for k = 1:numel(files)
    inname = files(k).name;                 % e.g. sub-001_task-xxx_preprocessed.set
    ID = extractBefore(inname, '_preprocessed.set');

    fprintf('\n=== Pre ICA for %s ===\n', inname);
    EEG = pop_loadset(inname);
    EEG = eeg_checkset(EEG);

    % pick channels to check:
    % priority 1: spreadsheet row for this ID (non NaN entries)
    % otherwise: all channels minus mastoids and ocular/bipolar by label
    if haveXLS
        row = T(strcmp(string(T.ID), string(ID)), :);
        if ~isempty(row)
            chans = table2array(row(:,2:end));
            chans = chans(~isnan(chans));
        else
            chans = derive_preica_chans_from_labels(EEG);
        end
    else
        chans = derive_preica_chans_from_labels(EEG);
    end

    % remove long segments without events (6000 ms, with 3000 ms buffers)
    EEG = pop_erplabDeleteTimeSegments(EEG, 'displayEEG', 0, ...
        'startEventcodeBufferMS', 3000, 'endEventcodeBufferMS', 3000, ...
        'timeThresholdMS', 6000);

    % extreme artifact detection on chosen channels (±300 uV, 500 ms window, 50 ms step)
    EEG = pop_continuousartdet(EEG, 'ampth', 300, 'winms', 500, 'stepms', 50, ...
        'chanArray', double(chans), 'review', 'off');

    % save
    setname = [ID '_preICA'];
    outname = [setname '.set'];
    EEG = pop_editset(EEG, 'setname', setname);
    pop_saveset(EEG, 'filename', outname, 'filepath', outDir);
    fprintf('Saved %s\n', fullfile(outDir, outname));
end

fprintf('\nAll done. Outputs in %s\n', outDir);
end

function chans = derive_preica_chans_from_labels(EEG)
% Build list of non ocular, non mastoid channels by label
nCh = size(EEG.data,1);
labs = lower(string({EEG.chanlocs.labels}));
allIdx = 1:nCh;

% exclude mastoids and oculars (including bipolar derivations if present)
exclude_labels = ["lm","m1","tp9","ma1","mastoidl","mastoid-left", ...
                  "rm","m2","tp10","ma2","mastoidr","mastoid-right", ...
                  "hel","her","ver","veog","veogr","heog", ...
                  "fp1","fp2"];  % authors drop Fp1/Fp2 for ICA prep
exIdx = find(ismember(labs, exclude_labels));

% also exclude typical bipolar channel indices if they exist
bipolar_guess = intersect([32 33], allIdx);
exIdx = union(exIdx, bipolar_guess);

chans = setdiff(allIdx, exIdx);
fprintf('Derived %d pre ICA channels (excluded %d)\n', numel(chans), numel(exIdx));
end

Writing IDEA_P3b_03_2_preICA.m


In [ ]:
eng.IDEA_P3b_03_2_preICA(nargout=0)

## script 5

In [ ]:
%%writefile IDEA_P3b_04_runICA.m
function IDEA_P3b_04_runICA
% Run ICA on pre ICA trimmed data:
% load *_preICA.set from 03_preICA
% choose ICA channels: all scalp except mastoids and bipolar derivations
% run extended infomax
% load *_preprocessed.set, inject weights, save to 04_ICAweighted

close all; clearvars;
eeglab nogui;

data_root  = '/content/drive/MyDrive/DataLiteracyProject';
project    = [data_root '/Preprocessed_VisualOddball/'];
preprocDir = [project '02_preprocessed/'];
preICADir  = [project '03_preICA/'];
outDir     = [project '04_ICAweighted/'];
if ~exist(outDir,'dir'); mkdir(outDir); end

cd(preICADir);
files = dir('*_preICA.set');
if isempty(files), error('No *_preICA.set in %s. Run the pre ICA step first.', preICADir); end

for k = 1:numel(files)
    preICAname = files(k).name;
    ID = extractBefore(preICAname, '_preICA.set');
    fprintf('\n=== ICA for %s ===\n', preICAname);

    % load pre ICA trimmed EEG
    EEG = pop_loadset(preICAname);
    EEG = eeg_checkset(EEG);

    % select ICA channels by label: keep scalp, drop mastoids and bipolar EOGs
    chanind = pick_ica_channels(EEG);

    % run extended infomax ICA
    EEG = pop_runica(EEG, 'extended', 1, 'chanind', double(chanind));

    % stash weights
    icachansind = EEG.icachansind;
    icaweights  = EEG.icaweights;
    icasphere   = EEG.icasphere;

    % load the continuous preprocessed EEG
    preprocName = [ID '_preprocessed.set'];
    EEG = pop_loadset('filename', preprocName, 'filepath', preprocDir);

    % transfer ICA weights
    EEG.icachansind = icachansind;
    EEG.icaweights  = icaweights;
    EEG.icasphere   = icasphere;

    % save weighted dataset
    setname = [ID 'ICA_weighted'];
    outname = [ID 'ICA_weighted.set'];
    EEG = pop_editset(EEG, 'setname', setname);
    pop_saveset(EEG, 'filename', outname, 'filepath', outDir);
    fprintf('Saved %s\n', fullfile(outDir, outname));
end

fprintf('\nAll done. Outputs in %s\n', outDir);
end

function chanind = pick_ica_channels(EEG)
nCh = size(EEG.data,1);
labs = lower(string({EEG.chanlocs.labels}));
allIdx = 1:nCh;

% typical exclusions: mastoids, bipolar EOGs if present
exclude_labels = ["lm","m1","tp9","ma1","mastoidl","mastoid-left", ...
                  "rm","m2","tp10","ma2","mastoidr","mastoid-right", ...
                  "veog","veogr","heog"]; % exclude bipolar derivations
exIdx = find(ismember(labs, exclude_labels));

bipolar_guess = intersect([32 33], allIdx);
exIdx = union(exIdx, bipolar_guess);

chanind = setdiff(allIdx, exIdx);
fprintf('ICA channels: %d kept, %d excluded\n', numel(chanind), numel(exIdx));
end

Overwriting IDEA_P3b_04_runICA.m


In [ ]:
eng.IDEA_P3b_04_runICA(nargout=0)

## script 6

In [ ]:
%%writefile IDEA_P3b_05_ICLabel.m
function IDEA_P3b_05_ICLabel
% ICLabel:
%  load *_ICA_weighted.set from 04_ICAweighted
%  run iclabel
%  save per-subject classifications to 05_ICLabel/*.xlsx
%  save per-subject PDF summary via pop_viewprops

close all; clearvars;
eeglab nogui;

% ensure ICLabel plugin is on the path
if exist('iclabel','file') ~= 2
    addpath(genpath('/content/eeglab/plugins/ICLabel'));
end

data_root = '/content/drive/MyDrive/DataLiteracyProject';
project   = [data_root '/Preprocessed_VisualOddball/'];
inDir     = [project '04_ICAweighted/'];
outDir    = [project '05_ICLabel/'];
if ~exist(outDir,'dir'); mkdir(outDir); end

cd(inDir);
files = dir('*ICA_weighted.set');
if isempty(files)
    error('No *ICA_weighted.set in %s. Run the ICA weighting step first.', inDir);
end

for k = 1:numel(files)
    setname = files(k).name;                          % e.g., sub-001_task-xxxICA_weighted.set
    ID = extractBefore(setname, 'ICA_weighted.set'); % e.g., sub-001_task-xxx
    fprintf('\n=== ICLabel for %s ===\n', setname);

    % load weighted dataset
    EEG = pop_loadset(setname);
    EEG = eeg_checkset(EEG);

    % run ICLabel
    EEG = iclabel(EEG);

    % classification matrix (Ncomp x 7): Brain, Muscle, Eye, Heart, Line Noise, Channel Noise, Other
    C = EEG.etc.ic_classification.ICLabel.classifications;
    C = round(C * 100);  % percent

    % to table with a Component column first
    comps = (1:size(C,1)).';
    T = array2table([comps, C], 'VariableNames', ...
        {'Component','Brain','Muscle','Eye','Heart','Line_Noise','Channel_Noise','Other'});

    % write spreadsheet
    xlsx_out = fullfile(outDir, [ID '_ICclassifications.xlsx']);
    writetable(T, xlsx_out);
    fprintf('Wrote %s\n', xlsx_out);

    % plot per-component properties (spectrum 2–40 Hz) and save to PDF
    try
        pop_viewprops(EEG, 0, 1:height(T), {'freqrange', [2 40]}, {}, 1, 'ICLabel');
        pdf_out = fullfile(outDir, [ID '_ICLabel.pdf']);
        print(gcf, '-dpdf', pdf_out, '-fillpage');
        fprintf('Wrote %s\n', pdf_out);
    catch ME
        warning('Could not render/save ICLabel PDF for %s: %s', ID, ME.message);
    end

    close all;
end

fprintf('\nAll done. Outputs in %s\n', outDir);
end


Writing IDEA_P3b_05_ICLabel.m


In [ ]:
eng.IDEA_P3b_05_ICLabel(nargout=0)
!ls -1 /content/project/COCOA_/05_ICLabel

ls: cannot access '/content/project/COCOA_/05_ICLabel': No such file or directory


## Script 7

In [ ]:
%%writefile IDEA_P3b_06_a_components_compiler.m
function IDEA_P3b_06_a_components_compiler
% Compile ICA eye components to be removed:
%  read each *_ICclassifications.xlsx from 05_ICLabel
%  mark components as "eye" if (Eye >=95) OR (Eye >=80 & Brain <=5)
%  write COCOA_ICA_eyecomponents.xlsx with two columns: ID, Components

close all; clearvars;

data_root = '/content/drive/MyDrive/DataLiteracyProject';
project   = [data_root '/Preprocessed_VisualOddball/'];
inDir     = [project '05_ICLabel/'];
outXLSX   = fullfile(inDir, 'COCOA_ICA_eyecomponents.xlsx');

if ~exist(inDir,'dir')
    error('Folder not found: %s', inDir);
end

cd(inDir);
files = dir('*_ICclassifications.xlsx');
if isempty(files)
    error('No *_ICclassifications.xlsx found in %s. Run the ICLabel step first.', inDir);
end

% Start grand table as a MATLAB table for robustness
Grand = table('Size', [0 2], 'VariableTypes', {'string','string'}, 'VariableNames', {'ID','Components'});

for k = 1:numel(files)
    fn = files(k).name;             % e.g., sub-001_task-xxx_ICclassifications.xlsx
    ID = extractBefore(fn, '_ICclassifications.xlsx');

    % load per-subject classifications
    T = readtable(fullfile(inDir, fn));
    % expected columns: Component, Brain, Muscle, Eye, Heart, Line_Noise, Channel_Noise, Other
    % tolerate case and underscores
    vn = lower(strrep(T.Properties.VariableNames,'__','_'));
    % locate needed columns
    cComp = find(strcmp(vn, 'component'), 1);
    cBrain= find(strcmp(vn, 'brain'), 1);
    cEye  = find(strcmp(vn, 'eye'), 1);
    if isempty(cComp) || isempty(cBrain) || isempty(cEye)
        warning('Skipping %s: missing expected columns.', fn);
        continue;
    end

    comps = T{:, cComp};
    brain = T{:, cBrain};
    eye   = T{:, cEye};

    % criteria:
    %  a) Eye >= 95
    %  b) Eye >= 80 AND Brain <= 5
    pick = (eye >= 95) | (eye >= 80 & brain <= 5);
    picked_list = comps(pick);

    if isempty(picked_list)
        comp_str = "NaN";
    else
        comp_str = strjoin(string(picked_list), ', ');
    end

    Grand = [Grand; {string(ID), comp_str}]; %#ok<AGROW>
end

% write (append creates dupes; we’ll overwrite for clean runs)
if exist(outXLSX,'file'), delete(outXLSX); end
writetable(Grand, outXLSX, 'WriteVariableNames', true);
fprintf('Wrote %s with %d row(s)\n', outXLSX, height(Grand));
end

Writing IDEA_P3b_06_a_components_compiler.m


In [ ]:
eng.IDEA_P3b_06_a_components_compiler(nargout=0)
!ls -1 /content/drive/MyDrive/DataLiteracyProject/Preprocessed_VisualOddball/05_ICLabel

COCOA_ICA_eyecomponents.xlsx
sub-001_task-visualoddball_eeg_ICclassifications.xlsx
sub-002_task-visualoddball_eeg_ICclassifications.xlsx
sub-003_task-visualoddball_eeg_ICclassifications.xlsx
sub-004_task-visualoddball_eeg_ICclassifications.xlsx
sub-005_task-visualoddball_eeg_ICclassifications.xlsx
sub-006_task-visualoddball_eeg_ICclassifications.xlsx
sub-007_task-visualoddball_eeg_ICclassifications.xlsx
sub-008_task-visualoddball_eeg_ICclassifications.xlsx
sub-009_task-visualoddball_eeg_ICclassifications.xlsx
sub-010_task-visualoddball_eeg_ICclassifications.xlsx
sub-011_task-visualoddball_eeg_ICclassifications.xlsx
sub-012_task-visualoddball_eeg_ICclassifications.xlsx
sub-013_task-visualoddball_eeg_ICclassifications.xlsx
sub-014_task-visualoddball_eeg_ICclassifications.xlsx
sub-015_task-visualoddball_eeg_ICclassifications.xlsx
sub-016_task-visualoddball_eeg_ICclassifications.xlsx
sub-017_task-visualoddball_eeg_ICclassifications.xlsx
sub-018_task-visualoddball_eeg_ICclassifications.xlsx

## script 8

In [ ]:
%%writefile IDEA_P3b_06_b_postICA.m
function IDEA_P3b_06_b_postICA
% Remove ICA components and create corrected bipolar EOG channels
% Inputs:
%   04_ICAweighted/*_ICA_weighted.set
%   05_ICLabel/COCOA_ICA_eyecomponents.xlsx  (ID, Components)
% Outputs:
%   06_postICA/*_postICA.set

close all; clearvars;
eeglab nogui;

data_root = '/content/drive/MyDrive/DataLiteracyProject';
project   = [data_root '/Preprocessed_VisualOddball/'];
inDir     = [project '04_ICAweighted/'];
labelDir  = [project '05_ICLabel/'];
outDir    = [project '06_postICA/'];

if ~exist(outDir,'dir'); mkdir(outDir); end

% read compiled components file
labelFile  = [labelDir 'COCOA_ICA_eyecomponents.xlsx'];
if exist(labelFile,'file') ~= 2
    error('Cannot find %s. Run the components compiler first.', labelFile);
end
CompTbl = readtable(labelFile);


cd(inDir);
files = dir('*ICA_weighted.set');
if isempty(files)
    error('No *ICA_weighted.set found in %s', inDir);
end

for k = 1:numel(files)
    setname = files(k).name;                       % e.g., sub-001_task-xxx_ICA_weighted.set
    ID = extractBefore(setname, 'ICA_weighted.set');
    fprintf('\n=== Post ICA for %s ===\n', setname);

    % load weighted dataset
    EEG = pop_loadset(setname);
    EEG = eeg_checkset(EEG);

    % find components to remove for this ID
    comps = find_components_for_id(CompTbl, ID);

    if isempty(comps)
        fprintf('No components to remove for %s. Skipping pop_subcomp.\n', ID);
    else
        fprintf('Removing components: %s\n', strjoin(string(comps), ', '));
        EEG = pop_subcomp(EEG, double(comps), 0);
        EEG = eeg_checkset(EEG);
    end

    % create ICA-corrected bipolar EOG channels using labels
    % CVEOGR = VER - FP2,  CHEOG = HER - HEL
    nCh = size(EEG.data,1);
    lab2idx = containers.Map(lower(string({EEG.chanlocs.labels})), num2cell(1:nCh));
    have = @(s) isKey(lab2idx, lower(string(s)));
    idx  = @(s) lab2idx(lower(string(s)));

    ops = {};
    next1 = nCh + 1;
    next2 = nCh + 2;

    if have('ver') && have('fp2')
        ops{end+1} = sprintf('ch%d = ch%d - ch%d label CVEOGR', next1, idx('ver'), idx('fp2')); %#ok<AGROW>
    else
        warning('Skipping CVEOGR. Need VER and FP2 labels.');
    end
    if have('her') && have('hel')
        ops{end+1} = sprintf('ch%d = ch%d - ch%d label CHEOG', next2, idx('her'), idx('hel')); %#ok<AGROW>
    else
        warning('Skipping CHEOG. Need HER and HEL labels.');
    end

    if ~isempty(ops)
        EEG = pop_eegchanoperator(EEG, ops);
        EEG = eeg_checkset(EEG);
    end

    % save post ICA
    EEG.setname = [ID 'postICA'];
    outname = [ID 'postICA.set'];
    pop_saveset(EEG, 'filename', outname, 'filepath', outDir);
    fprintf('Saved %s\n', fullfile(outDir, outname));
end

fprintf('\nAll done. Outputs in %s\n', outDir);
end

function comps = find_components_for_id(T, ID)
% T must have columns: ID, Components (comma separated or "NaN")
if ~ismember('ID', T.Properties.VariableNames) || ~ismember('Components', T.Properties.VariableNames)
    error('Expected columns ID and Components in compiled table.');
end
row = find(strcmpi(string(T.ID), string(ID)), 1);
if isempty(row)
    warning('ID %s not found in compiled table. No components will be removed.', ID);
    comps = [];
    return;
end
compStr = string(T.Components(row));
compStr = strtrim(compStr);
if compStr == "" || strcmpi(compStr, "NaN")
    comps = [];
    return;
end
parts = strtrim(split(compStr, ','));
nums  = str2double(parts);
comps = nums(~isnan(nums));
end


Writing IDEA_P3b_06_b_postICA.m


In [ ]:
eng.IDEA_P3b_06_b_postICA(nargout=0)
!ls -1 /content/drive/MyDrive/DataLiteracyProject/Preprocessed_VisualOddball/06_postICA

sub-001_task-visualoddball_eegpostICA.fdt
sub-001_task-visualoddball_eegpostICA.set
sub-002_task-visualoddball_eegpostICA.fdt
sub-002_task-visualoddball_eegpostICA.set
sub-003_task-visualoddball_eegpostICA.fdt
sub-003_task-visualoddball_eegpostICA.set
sub-004_task-visualoddball_eegpostICA.fdt
sub-004_task-visualoddball_eegpostICA.set
sub-005_task-visualoddball_eegpostICA.fdt
sub-005_task-visualoddball_eegpostICA.set
sub-006_task-visualoddball_eegpostICA.fdt
sub-006_task-visualoddball_eegpostICA.set
sub-007_task-visualoddball_eegpostICA.fdt
sub-007_task-visualoddball_eegpostICA.set
sub-008_task-visualoddball_eegpostICA.fdt
sub-008_task-visualoddball_eegpostICA.set
sub-009_task-visualoddball_eegpostICA.fdt
sub-009_task-visualoddball_eegpostICA.set
sub-010_task-visualoddball_eegpostICA.fdt
sub-010_task-visualoddball_eegpostICA.set
sub-011_task-visualoddball_eegpostICA.fdt
sub-011_task-visualoddball_eegpostICA.set
sub-012_task-visualoddball_eegpostICA.fdt
sub-012_task-visualoddball_eegpost

## script 9

In [ ]:
%%writefile IDEA_P3b_06_c_postICA_eventcodes.m
function IDEA_P3b_06_c_postICA_eventcodes
% Export EEG.event from post ICA sets to Excel

close all; clearvars;
eeglab nogui;

data_root = '/content/drive/MyDrive/DataLiteracyProject';
project   = [data_root '/Preprocessed_VisualOddball/'];
inDir     = [project '06_postICA/'];
outRoot   = [project '06_postICApracticeremoved/COCOA_postICA_eventcodes/'];

if ~exist(outRoot,'dir'); mkdir(outRoot); end

cd(inDir);
files = dir('*.set');
if isempty(files)
    error('No .set files found in %s. Run post ICA first.', inDir);
end

for k = 1:numel(files)
    setname = files(k).name;             % e.g., sub-001_task-xxx_postICA.set
    ID = extractBefore(setname, '.set');

    EEG = pop_loadset(setname);

    A = struct2table(EEG.event);
    outXLSX = fullfile(outRoot, [ID '_EEGevent.xlsx']);
    writetable(A, outXLSX);
    fprintf('Wrote %s\n', outXLSX);
end

fprintf('\nAll done. Event tables in %s\n', outRoot);
end


Writing IDEA_P3b_06_c_postICA_eventcodes.m


In [ ]:
eng.IDEA_P3b_06_c_postICA_eventcodes(nargout=0)
!ls -1 /content/drive/MyDrive/DataLiteracyProject/Preprocessed_VisualOddball/06_postICApracticeremoved/COCOA_postICA_eventcodes

sub-001_task-visualoddball_eegpostICA_EEGevent.xlsx
sub-002_task-visualoddball_eegpostICA_EEGevent.xlsx
sub-003_task-visualoddball_eegpostICA_EEGevent.xlsx
sub-004_task-visualoddball_eegpostICA_EEGevent.xlsx
sub-005_task-visualoddball_eegpostICA_EEGevent.xlsx
sub-006_task-visualoddball_eegpostICA_EEGevent.xlsx
sub-007_task-visualoddball_eegpostICA_EEGevent.xlsx
sub-008_task-visualoddball_eegpostICA_EEGevent.xlsx
sub-009_task-visualoddball_eegpostICA_EEGevent.xlsx
sub-010_task-visualoddball_eegpostICA_EEGevent.xlsx
sub-011_task-visualoddball_eegpostICA_EEGevent.xlsx
sub-012_task-visualoddball_eegpostICA_EEGevent.xlsx
sub-013_task-visualoddball_eegpostICA_EEGevent.xlsx
sub-014_task-visualoddball_eegpostICA_EEGevent.xlsx
sub-015_task-visualoddball_eegpostICA_EEGevent.xlsx
sub-016_task-visualoddball_eegpostICA_EEGevent.xlsx
sub-017_task-visualoddball_eegpostICA_EEGevent.xlsx
sub-018_task-visualoddball_eegpostICA_EEGevent.xlsx
sub-019_task-visualoddball_eegpostICA_EEGevent.xlsx
sub-020_task

## script 10

In [ ]:
%%writefile IDEA_P3b_06_c_postICA_practice_removed.m
function IDEA_P3b_06_c_postICA_practice_removed
% Remove practice at the beginning of each *_postICA set.
% Input:
%   /content/project/COCOA_/06_postICA/*.set
% Output:
%   /content/project/COCOA_/06_postICApracticeremoved/*_postICA_practiceremoved.set

close all; clearvars;
eeglab nogui;

data_root = '/content/drive/MyDrive/DataLiteracyProject';
project   = [data_root '/Preprocessed_VisualOddball/'];
inDir     = [project '06_postICA/'];
outDir    = [project '06_postICApracticeremoved/'];
if ~exist(outDir,'dir'); mkdir(outDir); end

% optional practice times file
timesXLSX = fullfile(outDir, 'COCOA_00_postICA_practice_times.xlsx');
haveTable = exist(timesXLSX,'file') == 2;
if haveTable
    T = readtable(timesXLSX);
    % be resilient to column naming
    if ismember('ID', T.Properties.VariableNames)
        idCol = 'ID';
    else
        idCol = T.Properties.VariableNames{1};
    end
    if ismember('endsec', lower(string(T.Properties.VariableNames)))
        % already has seconds
        endsecCol = T.Properties.VariableNames( ...
            find(strcmpi(T.Properties.VariableNames, 'endsec'), 1));
        T.endsec = T.(endsecCol{1});
    elseif ismember('SampleTime', T.Properties.VariableNames)
        T.endsec = T.SampleTime/500 + 1;  % 500 Hz plus 1 sec buffer
    else
        warning('Practice table lacks SampleTime or endsec. Will auto-detect from EEG.');
        haveTable = false;
    end
end

cd(inDir);
files = dir('*postICA.set');
if isempty(files), error('No *postICA.set in %s', inDir); end

for k = 1:numel(files)
    setname = files(k).name;                         % e.g., sub-001_task-visualoddball_postICA.set
    ID = extractBefore(setname, 'postICA.set');
    fprintf('\n=== Practice removal for %s ===\n', setname);

    EEG = pop_loadset(setname);
    EEG = eeg_checkset(EEG);

    if haveTable
        row = T(strcmp(string(T.(idCol)), string(setname)) | ...
                strcmp(string(T.(idCol)), string(ID)) | ...
                strcmp(string(T.(idCol)), string([ID 'postICA.set'])), :);
        if ~isempty(row) && ~isnan(row.endsec(1))
            endsec = row.endsec(1);
        else
            warning('No matching row found for %s in table. Auto-detecting practice end.', setname);
            endsec = auto_practice_end_seconds(EEG);
        end
    else
        endsec = auto_practice_end_seconds(EEG);
    end

    % drop everything up to practice end
    EEG = pop_select(EEG, 'notime', [0 endsec]);
    EEG = eeg_checkset(EEG);

    outName = [ID 'postICA_practiceremoved.set'];
    EEG = pop_editset(EEG, 'setname', [ID 'postICA_practiceremoved']);
    pop_saveset(EEG, 'filename', outName, 'filepath', outDir);
    fprintf('Saved %s\n', fullfile(outDir, outName));
end

fprintf('\nAll done. Outputs in %s\n', outDir);
end

function endsec = auto_practice_end_seconds(EEG)
% Heuristic for Visual Oddball: take the 10th stimulus code, then +1 s buffer.
stimSets = [11:15 21:25 31:35 41:45 51:55];
srate = EEG.srate;

% parse event types to numeric where possible
E = EEG.event;
types = strings(numel(E),1);
for i=1:numel(E), types(i) = string(E(i).type); end
numtypes = NaN(size(types));
for i=1:numel(types)
    x = str2double(types(i));
    if ~isnan(x), numtypes(i) = x; end
end

idxStim = find(ismember(numtypes, stimSets));
if numel(idxStim) >= 10
    lat = E(idxStim(10)).latency;  % samples
    endsec = double(lat)/double(srate) + 1.0;
    fprintf('Auto practice end at event %d (10th stimulus): %.3f s\n', idxStim(10), endsec);
else
    % fallback: first non-boundary, non-response event
    nonb = find(~strcmpi(types,'boundary'), 1, 'first');
    if isempty(nonb), nonb = 1; end
    lat = E(nonb).latency;
    endsec = double(lat)/double(srate) + 1.0;
    warning('Fewer than 10 stimulus events found. Using first event as practice end. %.3f s', endsec);
end
end


Overwriting IDEA_P3b_06_c_postICA_practice_removed.m


In [ ]:
eng.IDEA_P3b_06_c_postICA_practice_removed(nargout=0)
!ls -1 /content/drive/MyDrive/DataLiteracyProject/Preprocessed_VisualOddball//06_postICApracticeremoved

ls: cannot access '/content/project/COCOA_/06_postICApracticeremoved': No such file or directory


## script 11

Some matlab libraries for this section didn't work properly, so the way of dealing with this ended up rather messy. I should be able to make this more organized using the methods I used in script 13 which had a rather similar issue.

In [ ]:
%%writefile make_bdf_from_bids_events.m
function outPath = make_bdf_from_bids_events(events_root, outPath, task_pattern)
% Build an ERPLAB Binlister (BDF) from BIDS events.
% - Scans only matching *_events.tsv (optionally filtered by task_pattern)
% - Extracts numeric codes from the "value" column (handles 'S 12', 'S202', etc.)
% - Uses *_events.json value.Levels to drop boundary/response
% - Caches JSON parsing per folder to speed up multi-file scans
%
% Usage examples:
%   make_bdf_from_bids_events('/content/data/sub-014/eeg', OUT)          % one subject fast
%   make_bdf_from_bids_events('/content/data', OUT, 'visualoddball')     % all subjects, VO only

data_root = '/content/drive/MyDrive/DataLiteracyProject';

if nargin < 1 || isempty(events_root)
    % scan all subjects under OriginalData/sub-XXX/eeg
    events_root = fullfile(data_root, 'OriginalData');
end
if nargin < 2 || isempty(outPath)
    % central BDF under Preprocessed_VisualOddball
    outPath = fullfile(data_root, 'Preprocessed_VisualOddball', ...
                       'COCOA_00_scripts', 'VisualOddball_Bins_AllTrialTypes.txt');
end
if nargin < 3
    task_pattern = 'visualoddball';   % set '' to include all tasks
end


if isempty(task_pattern)
    glob = '*_events.tsv';
else
    glob = ['*' task_pattern '*_events.tsv'];
end

d = dir(fullfile(events_root, '**', glob));
if isempty(d)
    error('No matching events found under %s with pattern "%s"', events_root, glob);
end

codes = [];
labels = containers.Map('KeyType','double','ValueType','char');  % code -> description
json_cache = containers.Map('KeyType','char','ValueType','any'); % folder -> Levels map

for i = 1:numel(d)
    tsv = fullfile(d(i).folder, d(i).name);
    T = readtable(tsv, 'FileType','text', 'Delimiter','\t');

    % get "value" column case-insensitively
    vname = '';
    for nm = string(T.Properties.VariableNames)
        if strcmpi(nm, "value"), vname = char(nm); break; end
    end
    if isempty(vname)
        error('Missing "value" column in %s', tsv);
    end

    vals = T.(vname);
    nvals = extract_numeric(vals);
    codes = [codes; nvals(~isnan(nvals))]; %#ok<AGROW>

    % parse sibling JSON once per folder
    jf = strrep(tsv, '_events.tsv', '_events.json');
    lev_map = [];
    if exist(jf, 'file')
        key = d(i).folder;
        if isKey(json_cache, key)
            lev_map = json_cache(key);
        else
            meta = jsondecode(fileread(jf));
            if isfield(meta, 'value') && isfield(meta.value, 'Levels')
                lev = meta.value.Levels;
                lev_map = containers.Map('KeyType','double','ValueType','char');
                ks = fieldnames(lev);
                for kk = 1:numel(ks)
                    x = str2double(ks{kk});
                    if ~isnan(x), lev_map(x) = lev.(ks{kk}); end
                end
            end
            json_cache(key) = lev_map;
        end
    end

    % merge any labels we found
    if ~isempty(lev_map)
        lk = lev_map.keys;
        for kk = 1:numel(lk)
            code = lk{kk};
            labels(code) = lev_map(code);
        end
    end
end

u = unique(codes(:)');
keep = true(size(u));
for i = 1:numel(u)
    if isKey(labels, u(i))
        desc = lower(labels(u(i)));
        if contains(desc, 'boundary') || contains(desc, 'response')
            keep(i) = false;
        end
    end
end
u = sort(u(keep));

if isempty(u)
    error('No stimulus-like numeric event codes found.');
end

% ensure output folder
outdir = fileparts(outPath);
if ~exist(outdir,'dir'), mkdir(outdir); end

% write BDF
fid = fopen(outPath, 'w');
fprintf(fid, '; Auto-generated from BIDS events\n\n');
for i = 1:numel(u)
    code = u(i);
    if isKey(labels, code)
        raw = labels(code);
        lab = regexprep(raw, '[^A-Za-z0-9_]+', '_');
        lab = regexprep(lab, '_+', '_');
        if isempty(lab), lab = sprintf('EV_%d', code); end
    else
        lab = sprintf('EV_%d', code);
    end
    fprintf(fid, 'bin %d\n', i);
    fprintf(fid, '%s = %d\n\n', lab, code);
end
fclose(fid);
fprintf('Wrote %s with %d bins\n', outPath, numel(u));
end

function n = extract_numeric(vals)
if isnumeric(vals)
    n = double(vals(:));
elseif iscell(vals)
    n = nan(numel(vals),1);
    for k = 1:numel(vals)
        n(k) = parse_one(vals{k});
    end
elseif isstring(vals) || ischar(vals)
    s = string(vals);
    n = arrayfun(@(x) parse_one(x), s(:));
else
    n = nan(height(vals),1);
end
end

function x = parse_one(v)
if isnumeric(v), x = double(v); return; end
s = string(v);
m = regexp(s, '(-?\d+)', 'match', 'once');
if isempty(m), x = NaN; else, x = str2double(m); end
end


Writing make_bdf_from_bids_events.m


In [ ]:
DATA_ROOT = '/content/drive/MyDrive/DataLiteracyProject'
events_root = f'{DATA_ROOT}/OriginalData'
out_bdf = f'{DATA_ROOT}/Preprocessed_VisualOddball/COCOA_00_scripts/VisualOddball_Bins_AllTrialTypes.txt'

!mkdir -p "/content/drive/MyDrive/DataLiteracyProject/Preprocessed_VisualOddball/COCOA_00_scripts"
eng.addpath('/content', nargout=0)

# time the BDF build across all subjects
eng.eval(
    f"tic; make_bdf_from_bids_events('{events_root}','{out_bdf}','visualoddball'); "
    "t=toc; fprintf('BDF build: %.1f s\\n', t);",
    nargout=0
)

In [ ]:
%%writefile IDEA_P3b_07_epoch.m
function IDEA_P3b_07_epoch
% First half only:
%  load *_postICA_practiceremoved.set
%  shift stimulus codes by +20 ms for LCD delay
%  create and save ERPLAB event list text file
% No binning, no epoching here.

close all; clearvars;
eeglab nogui;

% make sure ERPLAB is on path
if exist('pop_creabasiceventlist','file') ~= 2
    addpath(genpath('/content/erplab'));
end

data_root = '/content/drive/MyDrive/DataLiteracyProject';
project   = [data_root '/Preprocessed_VisualOddball/'];
inDir  = [project '06_postICApracticeremoved/'];
outDir = [project '07_epoched/'];
if ~exist(outDir,'dir'); mkdir(outDir); end


% Binlister file is not used in this half, but keep the same existence check in case we might need it later
% bdf = fullfile(project, 'COCOA_00_scripts/VisualOddball_Bins_AllTrialTypes.txt');

% find VO inputs first, else any postICA_practiceremoved
files = dir(fullfile(inDir,'*task-visualoddball*postICA_practiceremoved.set'));
if isempty(files)
    error('No *task-visualoddball*postICA_practiceremoved.set in %s', inDir);
end

for k = 1:numel(files)
    setname = files(k).name;
    ID = extractBefore(setname, '_postICA_practiceremoved.set');
    fprintf('\n=== Preparing event list for %s ===\n', setname);

    % load continuous postICA_practiceremoved
    EEG = pop_loadset('filename', setname, 'filepath', inDir);
    EEG = eeg_checkset(EEG);

    % shift stimulus codes by 20 ms
    stimCodes = [11:15 21:25 31:35 41:45 51:55];
    EEG = pop_erplabShiftEventCodes(EEG, 'DisplayEEG', 0, ...
        'DisplayFeedback', 'summary', 'Eventcodes', stimCodes, ...
        'Rounding', 'earlier', 'Timeshift', 20);
    EEG = eeg_checkset(EEG);

    % write event list text file
    eventlist_txt = fullfile(outDir, [ID 'epoched_eventlist.txt']);
    EEG = pop_creabasiceventlist(EEG, 'AlphanumericCleaning','on', ...
        'BoundaryNumeric', {-99}, 'BoundaryString', {'boundary'}, ...
        'Eventlist', eventlist_txt);
    EEG = eeg_checkset(EEG);

    fprintf('Wrote event list: %s\n', eventlist_txt);
end

fprintf('\nFirst half complete. Event lists are in %s\n', outDir);
end


Writing IDEA_P3b_07_epoch.m


In [ ]:
eng.IDEA_P3b_07_epoch(nargout=0)
!ls -1 /content/drive/MyDrive/DataLiteracyProject/Preprocessed_VisualOddball/07_epoched

epoched_eventlist.txt


In [ ]:
# Epoch with MNE, hand off array to MATLAB via .mat, then QUICK EVENTLIST + save
!pip -q install mne pandas scipy

import os, re, glob, time
import numpy as np
import mne
from scipy.io import savemat
import matlab.engine

# paths
DATA_ROOT = '/content/drive/MyDrive/DataLiteracyProject'
PROJECT   = f'{DATA_ROOT}/Preprocessed_VisualOddball'
IN_DIR   = f'{PROJECT}/06_postICApracticeremoved'
BDF_PATH = f'{PROJECT}/COCOA_00_scripts/VisualOddball_Bins_AllTrialTypes.txt'
OUT_DIR  = f'{PROJECT}/07_epoched'
os.makedirs(OUT_DIR, exist_ok=True)


# choose VO files first; else any postICA_practiceremoved
files = sorted(glob.glob(os.path.join(IN_DIR, '*task-visualoddball*postICA_practiceremoved.set')))
assert files, f'No *task-visualoddball*postICA_practiceremoved.set inputs in {IN_DIR}'

# MATLAB engine + toolboxes
try:
    eng
except NameError:
    eng = matlab.engine.start_matlab("-licmode onlinelicensing")
eng.addpath('/content', nargout=0)
eng.addpath(eng.genpath('/content/eeglab'), nargout=0)
eng.addpath(eng.genpath('/content/erplab'), nargout=0)

# helpers: BDF parser and small profilers
import re

def parse_bdf_quick(path):
    code2bin = []
    cur_bin = None
    with open(path, encoding='utf-8', errors='ignore') as f:
        for raw in f:
            s = raw.strip()
            if not s or s.startswith(';'):
                continue
            m = re.match(r'bin\s+(\d+)', s, flags=re.I)
            if m:
                cur_bin = int(m.group(1)); continue
            if '=' in s and cur_bin is not None:
                rhs = s.split('=', 1)[1]
                for num in re.findall(r'-?\d+', rhs):
                    code2bin.append((int(num), cur_bin))
    return code2bin

def step(tag, cmd):
    t0 = time.time()
    out = eng.evalc(cmd)
    print(f'[{tag}] done in {time.time()-t0:.2f}s')
    if out.strip():
        print(out, end='')

def stepc(tag, cmd, timeout=120):
    print(f'[{tag}] start')
    fut = eng.eval_async(cmd)
    fut.result(timeout=timeout)  # raises TimeoutError if stuck
    print(f'[{tag}] done')

# parse BDF once
c2b = parse_bdf_quick(BDF_PATH)
assert c2b, f'No codes parsed from {BDF_PATH}'
# also keep the set of BDF codes to filter events fast
BDF_CODES = {c for c, _ in c2b}

# loop subjects
for fpath in files:
    print(f'\n=== Python second half on: {os.path.basename(fpath)}', flush=True)

    stem = os.path.splitext(os.path.basename(fpath))[0]
    stem = re.sub(r'(?:_eeg)?postICA_practiceremoved$', '', stem, flags=re.IGNORECASE)
    ID_BASE = stem + '_'

    # MNE: load continuous, build events, shift, filter, epoch
    print('Load raw...', flush=True)
    raw = mne.io.read_raw_eeglab(fpath, preload=True, verbose='ERROR')
    sfreq = float(raw.info['sfreq'])

    print('Build events and shift +20 ms for stim codes...', flush=True)
    events, _ = mne.events_from_annotations(raw, verbose=False)
    # shift only numeric stimulus codes that are in the BDF
    shift = int(round(0.02 * sfreq))
    mask = np.isin(events[:, 2], list(BDF_CODES))
    events[mask, 0] += shift

    # filter to BDF codes present in this file
    before = events.shape[0]
    keep_mask = np.isin(events[:, 2], list(BDF_CODES))
    events = events[keep_mask]
    after = events.shape[0]
    print(f'Filtered to BDF codes: {before} -> {after} events', flush=True)
    if after == 0:
        raise RuntimeError('No BDF-mapped events found after filtering')

    # build event_id from present codes
    present_codes = np.unique(events[:, 2]).tolist()
    event_id = {f'EV_{c}': int(c) for c in present_codes}

    print('Epoch with MNE...', flush=True)
    epochs = mne.Epochs(raw, events, event_id=event_id,
                        tmin=-0.2, tmax=0.8, baseline=(-0.2, 0.0),
                        preload=True, reject_by_annotation=True,
                        event_repeated='merge', verbose=False)

    # convert to ch x time x epochs and save uncompressed MAT for fast import
    X = epochs.get_data().astype('float32')
    X = np.transpose(X, (1, 2, 0))
    n_ch, n_times, n_ep = X.shape
    tmin = float(epochs.tmin)
    ev_codes = epochs.events[:, 2].astype('float64')  # one code per epoch

    tmp_mat = '/content/_tmp_epochs.mat'
    print(f'Write {tmp_mat} (uncompressed) ...', flush=True)
    savemat(tmp_mat, {'D3': X}, do_compression=False)

    # MATLAB: import D3, insert epoch events, build QUICK EVENTLIST from BDF mapping, save
    # share scalars and paths
    eng.workspace['tmp_mat']  = tmp_mat
    eng.workspace['n_ch']     = float(n_ch)
    eng.workspace['n_times']  = float(n_times)
    eng.workspace['n_ep']     = float(n_ep)
    eng.workspace['sfreq']    = float(sfreq)
    eng.workspace['tmin']     = float(tmin)
    eng.workspace['OUT_DIR']  = OUT_DIR
    eng.workspace['ID_BASE']  = ID_BASE
    eng.workspace['ORIG_SET'] = fpath
    eng.workspace['BDF_PATH'] = BDF_PATH
    eng.workspace['ev_codes'] = matlab.double(ev_codes.tolist())
    eng.workspace['code2bin'] = matlab.double([[c, b] for c, b in c2b])

    # load original set for chanlocs
    step('load_set',  r"[fp,fn,ext] = fileparts(ORIG_SET); EEG0 = pop_loadset('filename',[fn ext],'filepath',fp); fprintf('Loaded %s\n', [fn ext]);")

    # import array
    step('load_mat',  r"S = load(tmp_mat, 'D3'); D3 = S.D3; D3 = D3 * 1e6;")
    step('import',    r"EEG = eeg_emptyset; EEG = pop_importdata('dataformat','array','nbchan', n_ch, 'data', D3, 'srate', sfreq, 'pnts', n_times, 'xmin', tmin); EEG.chanlocs = EEG0.chanlocs;")

    # insert one event per epoch at time 0 carrying the epoch code
    step('insert_events', r"""
EEG.event = struct('type', {}, 'latency', {}, 'epoch', {});
t0 = round(-tmin*sfreq) + 1;
for e = 1:n_ep
    EEG.event(e).type    = ev_codes(e);
    EEG.event(e).latency = (e-1)*n_times + t0;
    EEG.event(e).epoch   = e;
end
EEG = eeg_checkset(EEG,'eventconsistency');
""")

    # quick EVENTLIST from BDF mapping (replaces pop_creabasiceventlist + pop_binlister)
    step('quick_eventlist', r"""
cb = code2bin; codes = cb(:,1); bins = cb(:,2);
K = num2cell(codes); V = num2cell(bins); map = containers.Map(K, V);

nep = EEG.trials;
evbin  = zeros(nep,1);
evcode = nan(nep,1);
for e = 1:nep
    idx = find([EEG.event.epoch] == e, 1, 'first');
    if isempty(idx), continue; end
    t = EEG.event(idx).type;
    if ischar(t), c = str2double(t); else, c = double(t); end
    evcode(e) = c;
    if ~isnan(c) && isKey(map, c), evbin(e) = map(c); else, evbin(e) = 0; end
end

nb = max([bins; 0]);
trialsperbin = zeros(1, nb);
for b = 1:nb
    trialsperbin(b) = sum(evbin==b);
end

EEG.EVENTLIST = struct();
EEG.EVENTLIST.trialsperbin = trialsperbin;
EEG.EVENTLIST.nbin         = nb;
EEG.EVENTLIST.eventbini    = evbin;
try, EEG.EVENTLIST.bdf = fileread(BDF_PATH); catch, EEG.EVENTLIST.bdf = BDF_PATH; end

EEG.EVENTLIST.eventinfo = struct('epoch', num2cell(1:nep), 'code', [], 'latency', [], 'bini', []);
for e = 1:nep
    idx = find([EEG.event.epoch]==e, 1, 'first');
    if ~isempty(idx)
        t = EEG.event(idx).type;
        if ischar(t), c = str2double(t); else, c = double(t); end
        lat = EEG.event(idx).latency;
    else
        c = NaN; lat = NaN;
    end
    EEG.EVENTLIST.eventinfo(e).epoch   = e;
    EEG.EVENTLIST.eventinfo(e).code    = c;
    EEG.EVENTLIST.eventinfo(e).latency = lat;
    EEG.EVENTLIST.eventinfo(e).bini    = evbin(e);
end
fprintf('Quick EVENTLIST built. nbin=%d, trialsperbin=%s\n', nb, mat2str(trialsperbin));
""")

    step('export_eventlist_txt', r"""
eventlist_txt = fullfile(OUT_DIR, [ID_BASE 'epoched_eventlist.txt']);
fid = fopen(eventlist_txt, 'w');
fprintf(fid, 'epoch\tcode\tbini\n');
for ii = 1:numel(EEG.EVENTLIST.eventinfo)
    fprintf(fid, '%d\t%d\t%d\n', ii, EEG.EVENTLIST.eventinfo(ii).code, EEG.EVENTLIST.eventinfo(ii).bini);
end
fclose(fid);
fprintf('Wrote %s\n', eventlist_txt);
""")

    # save the final epoched set in one file
    step('save_set', r"""
EEG = eeg_checkset(EEG);
setname = [ID_BASE 'epoched'];
outname = [ID_BASE 'epoched.set'];
EEG = pop_editset(EEG,'setname',setname);
pop_saveset(EEG,'filename',outname,'filepath',OUT_DIR,'savemode','onefile');
fprintf('Saved %s\n', fullfile(OUT_DIR, outname));
""")

    print('[Python] Done for', os.path.basename(fpath))
    !ls -lh /content/drive/MyDrive/DataLiteracyProject/Preprocessed_VisualOddball/07_epoched | sed -n '1,200p'


Streaming output truncated to the last 5000 lines.
-rw------- 1 root root  655 Nov 28 11:47 sub-049_task-visualoddball_epoched_eventlist.txt
-rw------- 1 root root 5.5M Nov 28 11:47 sub-049_task-visualoddball_epoched.set
-rw------- 1 root root  737 Nov 28 11:48 sub-050_task-visualoddball_epoched_eventlist.txt
-rw------- 1 root root 6.1M Nov 28 11:48 sub-050_task-visualoddball_epoched.set
-rw------- 1 root root  655 Nov 28 11:48 sub-051_task-visualoddball_epoched_eventlist.txt
-rw------- 1 root root 5.5M Nov 28 11:48 sub-051_task-visualoddball_epoched.set
-rw------- 1 root root  655 Nov 28 11:48 sub-052_task-visualoddball_epoched_eventlist.txt
-rw------- 1 root root 5.5M Nov 28 11:48 sub-052_task-visualoddball_epoched.set
-rw------- 1 root root  655 Nov 28 11:48 sub-053_task-visualoddball_epoched_eventlist.txt
-rw------- 1 root root 5.5M Nov 28 11:48 sub-053_task-visualoddball_epoched.set
-rw------- 1 root root  655 Nov 28 11:48 sub-054_task-visualoddball_epoched_eventlist.txt
-rw------

## script 12

In [ ]:
%%writefile IDEA_P3b_08_1_badchandetectpreAR.m
function IDEA_P3b_08_1_badchandetectpreAR
% Bad channel detection before Automatic Rejection
%  load *_epoched.set from 07_epoched
%  select channels from Excel if present, else derive from labels
%  amplitude check +/-200 µV within [-200 800] ms on each epoch
%  compute % loss in the bin with the most trials
%  mark channels: loss > 10 and z >= 3.29 as bad
%  write *_preARextremeloss.xlsx and *_ARchannels.xlsx to 08_AR

close all; clearvars;
eeglab nogui;

project  = '/content/drive/MyDrive/DataLiteracyProject/Preprocessed_VisualOddball/';
inDir    = [project '07_epoched/'];
outDir   = [project '08_AR/'];
bdfPath  = [project 'COCOA_00_scripts/VisualOddball_Bins_AllTrialTypes.txt'];
if ~exist(outDir,'dir'); mkdir(outDir); end

% optional channel list from Excel
excelPath = '/content/drive/MyDrive/DataLiteracyProject/IDEA_BV_channellocations.xlsx';
use_excel = exist(excelPath,'file') == 2;
if use_excel
    try
        chans = readtable(excelPath, 'Sheet', 'ARchans', 'NumHeaderLines', 0);
        if ~all(ismember({'Number','Label'}, chans.Properties.VariableNames))
            warning('ARchans sheet missing Number/Label. Deriving channels from EEG labels.');
            use_excel = false;
        end
    catch ME
        warning('Could not read ARchans sheet: %s. Deriving channels from EEG labels.', ME.message);
        use_excel = false;
    end
end

cd(inDir);
files = dir('*task-visualoddball*epoched.set');
if isempty(files)
    error('No *task-visualoddball*epoched.set in %s', inDir);
end

loss     = [];
ARchans  = [];
IDs      = strings(numel(files),1);
colLabels = [];

for k = 1:numel(files)
    setname = files(k).name;
    ID = extractBefore(setname, 'epoched.set');
    IDs(k) = string(ID);

    EEG = pop_loadset(setname);
    EEG = eeg_checkset(EEG);

    % Make sure EVENTLIST + bins exist
    if ~(isfield(EEG,'EVENTLIST') && isfield(EEG.EVENTLIST,'trialsperbin') && ~isempty(EEG.EVENTLIST.trialsperbin))
        if exist(bdfPath,'file') ~= 2
            error('BDF not found at %s. Generate it first.', bdfPath);
        end
        EEG = pop_creabasiceventlist(EEG, 'AlphanumericCleaning','off', ...
            'BoundaryNumeric', {-99}, 'BoundaryString', {'boundary'}, 'Eventlist','');
        EEG = pop_binlister(EEG, 'BDF', bdfPath, 'IndexEL', 1, 'SendEL2','EEG','Voutput','EEG');
    end

    % Epoch-to-bin vector from EVENTLIST.eventinfo(i).bini
    ep2bin = local_epoch_to_bin(EEG);  % 1 x nEpochs
    trialsperbin = EEG.EVENTLIST.trialsperbin(:).';
    if numel(trialsperbin) < 1
        error('No bins found in %s', setname);
    end
    [~, binIdx] = max(trialsperbin);

    % channel list
    if use_excel
        chNums   = chans.Number(:).';
        chLabels = string(chans.Label(:)).';
    else
        [chNums, chLabels] = local_nonocular_mastoid_channels(EEG);
    end
    nCh = numel(chNums);
    if k == 1, colLabels = chLabels; end

    % analysis window indices in samples
    widx = (EEG.times >= -200) & (EEG.times <= 800);
    if ~any(widx), error('Window [-200 800] ms not found in EEG.times'); end

    % amplitude threshold in microvolts
    thr = 200;

    % compute rejects per channel in selected bin
    loss_this = nan(1, nCh);
    for j = 1:nCh
        cj = chNums(j);
        % max absolute amplitude per epoch within window
        % data dims ch x points x epochs
        x = squeeze(EEG.data(cj, widx, :));        % points x epochs
        if isvector(x), x = x(:).'; end
        maxabs = max(abs(x), [], 1);               % 1 x epochs
        rej = maxabs > thr;                        % 1 x epochs logical
        % count rejects in the chosen bin
        loss_this(j) = sum(rej & (ep2bin == binIdx));
    end

    bintotal = trialsperbin(binIdx);
    loss_pct = loss_this / bintotal * 100;

    z = local_zscore(loss_pct);

    include = nan(1, nCh);
    for j = 1:nCh
        if loss_pct(j) <= 10 && z(j) < 3.29
            include(j) = chNums(j);
        end
    end

    loss(k, 1:nCh)    = loss_pct;
    ARchans(k, 1:nCh) = include;
end

% tables
loss_tbl = array2table(loss(:, 1:numel(colLabels)));
loss_tbl.Properties.VariableNames = matlab.lang.makeValidName(colLabels);
loss_tbl = addvars(loss_tbl, IDs, 'Before', 1, 'NewVariableNames', 'ID');

arch_tbl = array2table(ARchans(:, 1:numel(colLabels)));
arch_tbl.Properties.VariableNames = matlab.lang.makeValidName(colLabels);
arch_tbl = addvars(arch_tbl, IDs, 'Before', 1, 'NewVariableNames', 'ID');

proj = 'COCOA_VO';
outLoss = fullfile(outDir, [proj '_preARextremeloss.xlsx']);
outArch = fullfile(outDir, [proj '_ARchannels.xlsx']);

if exist(outLoss,'file'), writetable(loss_tbl, outLoss, 'WriteMode','append');
else, writetable(loss_tbl, outLoss); end

if exist(outArch,'file'), writetable(arch_tbl, outArch, 'WriteMode','append');
else, writetable(arch_tbl, outArch); end

fprintf('Wrote:\n  %s\n  %s\n', outLoss, outArch);
end

function ep2bin = local_epoch_to_bin(EEG)
% Build 1 x nEpochs vector mapping each epoch to its first bin assignment.
nep = EEG.trials;
ep2bin = zeros(1, nep);  % 0 means unassigned
if isfield(EEG,'EVENTLIST') && isfield(EEG.EVENTLIST,'eventinfo')
    EI = EEG.EVENTLIST.eventinfo;
    for i = 1:numel(EI)
        if isfield(EI(i),'epoch') && ~isempty(EI(i).epoch) && EI(i).epoch>0 ...
           && isfield(EI(i),'bini')  && ~isempty(EI(i).bini)
            ep = EI(i).epoch;
            if ep<=nep && ep2bin(ep)==0
                ep2bin(ep) = EI(i).bini(1);
            end
        end
    end
end
end

function [nums, labels] = local_nonocular_mastoid_channels(EEG)
nCh = size(EEG.data,1);
labels = string({EEG.chanlocs.labels});
labs = lower(labels);
exclude = ["lm","m1","tp9","ma1","mastoidl","mastoid-left", ...
           "rm","m2","tp10","ma2","mastoidr","mastoid-right", ...
           "hel","her","ver","veog","veogr","heog","cveogr","cheog"];
keep = find(~ismember(labs, exclude));
nums = keep(:).';
labels = labels(keep).';
end

function z = local_zscore(x)
x = double(x);
mu = mean(x, 'omitnan');
sd = std(x, 0, 'omitnan');
if sd == 0 || isnan(sd)
    z = zeros(size(x));
else
    z = (x - mu) ./ sd;
end
end


Writing IDEA_P3b_08_1_badchandetectpreAR.m


In [ ]:
p = eng.genpath('/content/erplab'); eng.addpath(p, nargout=0)
eng.IDEA_P3b_08_1_badchandetectpreAR(nargout=0)
!ls -1 /content/drive/MyDrive/DataLiteracyProject/Preprocessed_VisualOddball/08_AR

COCOA_VO_ARchannels.xlsx
COCOA_VO_preARextremeloss.xlsx


## script 13

In [ ]:
%%writefile IDEA_P3b_08_2_autoAR.m
function IDEA_P3b_08_2_autoAR
% Automatic artifact rejection for Visual Oddball (manual implementation)
% read AR channels from 08_AR/COCOA_VO_ARchannels.xlsx
% ensure EVENTLIST + bins exist (using BDF map if needed)
% run amplitude, moving-window p2p, blink and eye-movement tests by hand
% save *_autoAR.set and append AR results table

close all; clearvars;
eeglab nogui;

project = '/content/drive/MyDrive/DataLiteracyProject/Preprocessed_VisualOddball/';
inDir   = [project '07_epoched/'];
outDir  = [project '08_AR/'];
bdfPath = [project 'COCOA_00_scripts/VisualOddball_Bins_AllTrialTypes.txt'];
if ~exist(outDir,'dir'); mkdir(outDir); end

proj = 'COCOA_VO';
archPath = fullfile(outDir, [proj '_ARchannels.xlsx']);
if exist(archPath,'file') ~= 2
    error('AR channels file not found: %s', archPath);
end
ARchannels = readtable(archPath);

% Parse BDF once to a code->bin map (for safety if we need to rebuild EVENTLIST)
if exist(bdfPath,'file') ~= 2
    error('BDF not found at %s', bdfPath);
end
code2bin = local_parse_bdf_quick(bdfPath);

cd(inDir);
files = dir('*task-visualoddball*epoched.set');
if isempty(files)
    error('No *task-visualoddball*epoched.set in %s', inDir);
end

ARsummary = cell(numel(files), 1);

for k = 1:numel(files)
    setname = files(k).name;
    ID = extractBefore(setname, 'epoched.set');
    fprintf('\n=== Auto AR (manual) for %s ===\n', setname);

    t0 = tic;
    EEG = pop_loadset(setname);
    EEG = eeg_checkset(EEG);
    EEG.history = '';
    fprintf('Load: %.2fs\n', toc(t0));

    % Ensure EVENTLIST has trialsperbin and eventbini
    EEG = local_ensure_eventlist(EEG, code2bin, bdfPath);

    % Bin with most trials
    [~, binIdx] = max(EEG.EVENTLIST.trialsperbin(:));
    epbin = EEG.EVENTLIST.eventbini(:);   % epoch -> bin

    % Channels from spreadsheet else derive from labels
    firstCol = string(ARchannels{:,1});
    ridx = find(strcmpi(firstCol, string(ID)) | strcmpi(firstCol, string(setname)), 1);
    if isempty(ridx)
        warning('No AR channel row for %s. Using derived scalp channels.', ID);
        [Channels, ~] = local_nonocular_mastoid_channels(EEG);
    else
        row = ARchannels(ridx, :);
        chans_vec = table2array(row(:, 2:end));
        Channels = double(chans_vec(~isnan(chans_vec))).';
        if isempty(Channels)
            [Channels, ~] = local_nonocular_mastoid_channels(EEG);
        end
    end

    % EOG indices by label
    chVEOGR = find_label_idx(EEG, ["VEOGR","VEOG"]);
    chHEOG  = find_label_idx(EEG, ["HEOG"]);
    chCVEOG = find_label_idx(EEG, ["CVEOGR"]);
    chCHEOG = find_label_idx(EEG, ["CHEOG"]);

    % Run artifact tests manually
    fprintf('Manual AR tests...\n');
    [rej1, rej2, rej3, rej4, rej5, rej6] = local_compute_artifacts( ...
        EEG, Channels, chVEOGR, chHEOG, chCVEOG, chCHEOG);

    % Summarize counts for the busiest bin (same logic as original script)
    counts = nan(1,6);
    if ~isempty(binIdx) && binIdx > 0
        counts(1) = sum(rej1(:) & (epbin == binIdx)); % AR_Chanart1
        counts(2) = sum(rej2(:) & (epbin == binIdx)); % AR_Chanart2
        counts(3) = sum(rej3(:) & (epbin == binIdx)); % AR_VEBlink
        counts(4) = sum(rej4(:) & (epbin == binIdx)); % AR_HEBlink
        counts(5) = sum(rej5(:) & (epbin == binIdx)); % AR_HEremaining
        counts(6) = sum(rej6(:) & (epbin == binIdx)); % AR_Blinkremaining
    end

    loss = array2table(counts, ...
        'VariableNames', {'AR_Chanart1','AR_Chanart2','AR_VEBlink','AR_HEBlink','AR_HEremaining','AR_Blinkremaining'});
    loss = addvars(loss, string(ID), 'Before', 1, 'NewVariableNames', 'ID');
    ARsummary{k} = loss;

    % Mark combined rejection in EEG.reject for downstream scripts
    rej_combined = rej1 | rej2 | rej3 | rej4 | rej5 | rej6;
    EEG.reject.rejmanual  = logical(rej_combined(:)).';
    EEG.reject.rejmanualE = logical(rej_combined(:)).'; % common EEGLAB pattern
    EEG.reject.rejglobal  = EEG.reject.rejmanual;
    EEG.reject.rejglobalE = EEG.reject.rejmanualE;

    % export rejglobal as a small sidecar MAT file for Feature Extraction
    rejglobal = EEG.reject.rejglobal;
    save(fullfile(outDir, [ID 'autoAR_rejglobal.mat']), 'rejglobal');

    EEG = pop_editset(EEG, 'setname', [ID 'autoAR']);
    pop_saveset(EEG, 'filename', [ID 'autoAR.set'], 'filepath', outDir);

    EEG = pop_editset(EEG, 'setname', [ID 'autoAR']);
    pop_saveset(EEG, 'filename', [ID 'autoAR.set'], 'filepath', outDir);
    EEG = eeg_checkset(EEG);
    fprintf('Saved %s (total %.2fs)\n', fullfile(outDir, [ID 'autoAR.set']), toc(t0));
end

ARresults = vertcat(ARsummary{:});
outRes = fullfile(outDir, [proj '_ARresults.xlsx']);
if exist(outRes,'file')
    writetable(ARresults, outRes, 'WriteMode','append');
else
    writetable(ARresults, outRes);
end
fprintf('Saved AR results to %s\n', outRes);
end

% ---------- helpers ----------

function idx = find_label_idx(EEG, candidates)
labs = lower(string({EEG.chanlocs.labels}));
idx = [];
for c = lower(string(candidates))
    f = find(labs == c, 1);
    if ~isempty(f), idx = f; return; end
end
end

function [nums, labels] = local_nonocular_mastoid_channels(EEG)
nCh = size(EEG.data,1);
labels = string({EEG.chanlocs.labels});
labs = lower(labels);
exclude = ["lm","m1","tp9","ma1","mastoidl","mastoid-left", ...
           "rm","m2","tp10","ma2","mastoidr","mastoid-right", ...
           "hel","her","ver","veog","veogr","heog","cveogr","cheog"];
keep = find(~ismember(labs, exclude));
nums = keep(:).';
labels = labels(keep).';
end

function map = local_parse_bdf_quick(bdfPath)
fid = fopen(bdfPath, 'r');
if fid < 0, error('Cannot open BDF %s', bdfPath); end
c = onCleanup(@() fclose(fid));
map = containers.Map('KeyType','double','ValueType','double');
curbin = NaN;
while true
    ln = fgetl(fid);
    if ~ischar(ln), break; end
    s = strtrim(ln);
    if isempty(s) || strncmp(s,';',1), continue; end
    tok = regexp(s, '^bin\s+(\d+)', 'tokens', 'once', 'ignorecase');
    if ~isempty(tok)
        curbin = str2double(tok{1});
        continue;
    end
    if ~isnan(curbin) && contains(s, '=')
        rhs = strtrim(s(find(s=='=',1,'first')+1:end));
        nums = regexp(rhs, '(-?\d+)', 'match');
        for i = 1:numel(nums)
            code = str2double(nums{i});
            if ~isnan(code), map(code) = curbin; end
        end
    end
end
end

function EEG = local_ensure_eventlist(EEG, code2bin, bdfPath)
% Ensure EVENTLIST.trialsperbin and EVENTLIST.eventbini exist.
if isfield(EEG,'EVENTLIST') && ...
   isfield(EEG.EVENTLIST,'trialsperbin') && ...
   isfield(EEG.EVENTLIST,'eventbini') && ...
   numel(EEG.EVENTLIST.eventbini) == EEG.trials
    return;
end

fprintf('EVENTLIST missing or incomplete. Rebuilding from BDF map.\n');

nep = EEG.trials;
evbin  = zeros(nep,1);
evcode = nan(nep,1);
evlat  = nan(nep,1);

for e = 1:nep
    idx = find([EEG.event.epoch] == e, 1, 'first');
    if isempty(idx), continue; end
    t = EEG.event(idx).type;
    if ischar(t), c = str2double(t); else, c = double(t); end
    evcode(e) = c;
    evlat(e)  = EEG.event(idx).latency;
    if ~isnan(c) && isKey(code2bin, c)
        evbin(e) = code2bin(c);
    else
        evbin(e) = 0;
    end
end

nb = 0;
if code2bin.Count > 0
    vv = values(code2bin);
    nb = max(cell2mat(vv));
end
trialsperbin = zeros(1, nb);
for b = 1:nb
    trialsperbin(b) = sum(evbin == b);
end

EEG.EVENTLIST = struct();
EEG.EVENTLIST.trialsperbin = trialsperbin;
EEG.EVENTLIST.nbin         = nb;
EEG.EVENTLIST.eventbini    = evbin;
try, EEG.EVENTLIST.bdf = fileread(bdfPath); catch, EEG.EVENTLIST.bdf = bdfPath; end

EEG.EVENTLIST.eventinfo = repmat(struct('epoch', [], 'code', [], 'latency', [], 'bini', []), 1, nep);
for e = 1:nep
    EEG.EVENTLIST.eventinfo(e).epoch   = e;
    EEG.EVENTLIST.eventinfo(e).code    = evcode(e);
    EEG.EVENTLIST.eventinfo(e).latency = evlat(e);
    EEG.EVENTLIST.eventinfo(e).bini    = evbin(e);
end
end

function [rej1, rej2, rej3, rej4, rej5, rej6] = local_compute_artifacts(EEG, Channels, chVEOGR, chHEOG, chCVEOG, chCHEOG)
% Manual implementation of 6 artifact tests

nEp   = EEG.trials;
times = EEG.times;
data  = EEG.data;

rej1 = false(1, nEp);
rej2 = false(1, nEp);
rej3 = false(1, nEp);
rej4 = false(1, nEp);
rej5 = false(1, nEp);
rej6 = false(1, nEp);

% 1) Amplitude threshold on scalp: [-200 800] ms, ±200 uV
twin = [-200 800];
idx  = find(times >= twin(1) & times <= twin(2));
if ~isempty(idx) && ~isempty(Channels)
    d = data(Channels, idx, :);                 % ch x time x ep
    d2 = reshape(d, [], nEp);                   % (ch*time) x ep
    dmax = max(d2, [], 1);
    dmin = min(d2, [], 1);
    rej1 = (dmax > 200) | (dmin < -200);
end

% 2) Moving-window p2p on scalp: [-200 800], thresh 125, win 1000 ms, step 100 ms
if ~isempty(Channels)
    rej2 = local_mwpp(data, times, Channels, [-200 800], 1000, 100, 125);
end

% 3) Blink window VEOGR: [-25 225], thresh 150, win 175 ms, step 10 ms
if ~isempty(chVEOGR)
    rej3 = local_mwpp(data, times, chVEOGR, [-25 225], 175, 10, 150);
end

% 4) HEOG step stim window: [-50 250], thresh 32, win 100 ms, step 10 ms
if ~isempty(chHEOG)
    rej4 = local_step(data, times, chHEOG, [-50 250], 100, 10, 32);
end

% 5) CHEOG step full window: [-200 800], thresh 64, win 100 ms, step 10 ms
if ~isempty(chCHEOG)
    rej5 = local_step(data, times, chCHEOG, [-200 800], 100, 10, 64);
end

% 6) CVEOGR p2p full window: [-200 800], thresh 150, win 200 ms, step 50 ms
if ~isempty(chCVEOG)
    rej6 = local_mwpp(data, times, chCVEOG, [-200 800], 200, 50, 150);
end
end

function rej = local_mwpp(data, times, chanIdx, Twindow, winsize_ms, step_ms, thresh)
% Moving-window peak to peak detection
%
% data: ch x time x ep
% times: 1 x time (ms)
% chanIdx: vector of channels
% Twindow: [start end] in ms
% winsize_ms: window size in ms
% step_ms: window step in ms
% thresh: p2p threshold

[~, nTime, nEp] = size(data);

maskTW = times >= Twindow(1) & times <= Twindow(2);
if ~any(maskTW)
    rej = false(1, nEp);
    return;
end

win_starts = Twindow(1):step_ms:(Twindow(2) - winsize_ms);
if isempty(win_starts)
    win_starts = Twindow(1);
end
nWin = numel(win_starts);

win_idx = cell(1, nWin);
for w = 1:nWin
    wstart = win_starts(w);
    wend   = wstart + winsize_ms;
    maskW  = times >= wstart & times <= wend;
    win_idx{w} = find(maskW);
end

rej = false(1, nEp);

for e = 1:nEp
    for c = 1:numel(chanIdx)
        ch = chanIdx(c);
        sig = squeeze(data(ch, :, e));  % time x 1
        for w = 1:nWin
            idx = win_idx{w};
            if isempty(idx), continue; end
            seg = sig(idx);
            p2p = max(seg) - min(seg);
            if p2p > thresh
                rej(e) = true;
                break;
            end
        end
        if rej(e), break; end
    end
end
end

function rej = local_step(data, times, chanIdx, Twindow, winsize_ms, step_ms, thresh)
% Moving-window step detection based on max abs diff

[~, nTime, nEp] = size(data);

maskTW = times >= Twindow(1) & times <= Twindow(2);
if ~any(maskTW)
    rej = false(1, nEp);
    return;
end

win_starts = Twindow(1):step_ms:(Twindow(2) - winsize_ms);
if isempty(win_starts)
    win_starts = Twindow(1);
end
nWin = numel(win_starts);

win_idx = cell(1, nWin);
for w = 1:nWin
    wstart = win_starts(w);
    wend   = wstart + winsize_ms;
    maskW  = times >= wstart & times <= wend;
    win_idx{w} = find(maskW);
end

rej = false(1, nEp);

for e = 1:nEp
    for c = 1:numel(chanIdx)
        ch = chanIdx(c);
        sig = squeeze(data(ch, :, e));  % time x 1
        for w = 1:nWin
            idx = win_idx{w};
            if isempty(idx), continue; end
            seg = sig(idx);
            dv  = diff(seg);
            stepval = max(abs(dv));
            if stepval > thresh
                rej(e) = true;
                break;
            end
        end
        if rej(e), break; end
    end
end
end


Writing IDEA_P3b_08_2_autoAR.m


In [ ]:
eng.addpath('/content', nargout=0)
p = eng.genpath('/content/erplab'); eng.addpath(p, nargout=0)
eng.IDEA_P3b_08_2_autoAR(nargout=0)
!ls -1 /content/drive/MyDrive/DataLiteracyProject/Preprocessed_VisualOddball/08_AR

COCOA_VO_ARchannels.xlsx
COCOA_VO_ARresults.xlsx
COCOA_VO_preARextremeloss.xlsx
sub-001_task-visualoddball_autoAR.fdt
sub-001_task-visualoddball_autoAR_rejglobal.mat
sub-001_task-visualoddball_autoAR.set
sub-002_task-visualoddball_autoAR.fdt
sub-002_task-visualoddball_autoAR_rejglobal.mat
sub-002_task-visualoddball_autoAR.set
sub-003_task-visualoddball_autoAR.fdt
sub-003_task-visualoddball_autoAR_rejglobal.mat
sub-003_task-visualoddball_autoAR.set
sub-004_task-visualoddball_autoAR.fdt
sub-004_task-visualoddball_autoAR_rejglobal.mat
sub-004_task-visualoddball_autoAR.set
sub-005_task-visualoddball_autoAR.fdt
sub-005_task-visualoddball_autoAR_rejglobal.mat
sub-005_task-visualoddball_autoAR.set
sub-006_task-visualoddball_autoAR.fdt
sub-006_task-visualoddball_autoAR_rejglobal.mat
sub-006_task-visualoddball_autoAR.set
sub-007_task-visualoddball_autoAR.fdt
sub-007_task-visualoddball_autoAR_rejglobal.mat
sub-007_task-visualoddball_autoAR.set
sub-008_task-visualoddball_autoAR.fdt
sub-008_task-v

## VisualOddball P3b Feature Extraction

In [ ]:
# Install dependencies (Colab)
!pip -q install mne pandas scipy

import os
import glob
import re
import json
import numpy as np
import pandas as pd
import mne
from scipy.io import loadmat

# CONFIG
PROJECT = "/content/drive/MyDrive/DataLiteracyProject/Preprocessed_VisualOddball"
AUTO_DIR = os.path.join(PROJECT, "08_AR")
BDF_PATH = os.path.join(PROJECT, "COCOA_00_scripts", "VisualOddball_Bins_AllTrialTypes.txt")

BIDS_EVENTS_DIR = "/content/drive/MyDrive/DataLiteracyProject/OriginalData"
PARTICIPANTS_TSV = os.path.join(BIDS_EVENTS_DIR, "participants.tsv")

P3B_WINDOW = (0.300, 0.600)  # seconds
P3B_CH = "Pz"

# Baseline correction (standard ERP practice)
# Only applied if epochs.baseline is None
BASELINE = (-0.200, 0.000)

# Output
OUT_TRIAL_CSV = os.path.join(PROJECT, "COCOA_VO_P3b_trial_features_with_meta.csv")
OUT_SUBJ_CSV  = os.path.join(PROJECT, "COCOA_VO_P3b_subject_features_with_meta.csv")


# HELPERS
def find_events_json_for_subject(sub_id, task="visualoddball"):
    candidates = [
        os.path.join(BIDS_EVENTS_DIR, f"{sub_id}_task-{task}_events.json"),
        os.path.join(BIDS_EVENTS_DIR, sub_id, "eeg", f"{sub_id}_task-{task}_events.json"),
        os.path.join(BIDS_EVENTS_DIR, sub_id, f"{sub_id}_task-{task}_events.json"),
    ]
    for p in candidates:
        if os.path.exists(p):
            return p
    pattern = os.path.join(BIDS_EVENTS_DIR, "**", f"{sub_id}_task-{task}_events.json")
    hits = glob.glob(pattern, recursive=True)
    if hits:
        return hits[0]
    raise FileNotFoundError(f"Could not find events JSON for {sub_id} task {task} under {BIDS_EVENTS_DIR}")

def get_target_and_standard_codes(events_json_path):
    """
    Parse *_events.json to decide which numeric codes are targets vs standards.
    Uses text like:
      'Target (for that block) B, stimulus type A'
    If block letter == stimulus letter => target, else standard.
    """
    with open(events_json_path, "r") as f:
        meta = json.load(f)

    levels = meta["value"]["Levels"]
    target_codes = []
    standard_codes = []

    for code_key, desc in levels.items():
        # Robustly parse the numeric code from the JSON key
        mcode = re.search(r"-?\d+", str(code_key))
        if not mcode:
            continue
        code = int(mcode.group(0))

        # Skip non-stim codes (RT etc) if present
        if code >= 200:
            continue

        # Case-insensitive parsing of letters
        m1 = re.search(r"target\s*\(for that block\)\s*([A-Za-z])", str(desc), flags=re.I)
        m2 = re.search(r"stimulus type\s*([A-Za-z])", str(desc), flags=re.I)
        if not (m1 and m2):
            continue

        block_letter = m1.group(1).upper()
        stim_letter  = m2.group(1).upper()

        if block_letter == stim_letter:
            target_codes.append(code)
        else:
            standard_codes.append(code)

    target_codes = sorted(set(target_codes))
    standard_codes = sorted(set(standard_codes))

    if not target_codes:
        raise RuntimeError(f"No target codes found in {events_json_path}. Check the JSON format or adjust parsing.")

    # Sanity check
    overlap = set(target_codes).intersection(set(standard_codes))
    if overlap:
        print(f"WARNING: target/standard code overlap detected in {events_json_path}: {sorted(list(overlap))}")

    return target_codes, standard_codes

def parse_bdf_code2bin(bdf_path):
    code2bin = {}
    cur_bin = None
    with open(bdf_path, "r") as f:
        for line in f:
            s = line.strip()
            if not s or s.startswith(";"):
                continue
            mb = re.match(r"bin\s+(\d+)", s, flags=re.I)
            if mb:
                cur_bin = int(mb.group(1))
                continue
            if cur_bin is None or "=" not in s:
                continue
            rhs = s.split("=", 1)[1]
            for num in re.findall(r"-?\d+", rhs):
                code2bin[int(num)] = cur_bin
    return code2bin

def load_rejglobal_from_set(set_path):
    base, fname = os.path.split(set_path)
    root, _ = os.path.splitext(fname)
    rej_path = os.path.join(base, root + "_rejglobal.mat")

    if not os.path.exists(rej_path):
        print(f"  Warning: {rej_path} not found")
        return None

    try:
        mat = loadmat(rej_path)
    except Exception as e:
        print(f"  Warning: could not load {rej_path}: {e}")
        return None

    if "rejglobal" not in mat:
        print(f"  Warning: rejglobal missing in {rej_path}")
        return None

    return np.asarray(mat["rejglobal"]).astype(bool).ravel()

def decode_trial_codes_from_mne(epochs):
    """
    Decode numeric event codes per trial from an MNE Epochs object.
    We invert epochs.event_id to get index -> label, then parse the first integer from the label.
    """
    id_to_label = {v: k for k, v in epochs.event_id.items()}

    out = np.full((len(epochs.events),), np.nan, dtype=float)
    for i, ev_id in enumerate(epochs.events[:, 2]):
        label = id_to_label.get(ev_id, None)
        if label is None:
            continue
        if isinstance(label, (int, np.integer)):
            out[i] = int(label)
            continue
        m = re.search(r"-?\d+", str(label))
        if m:
            out[i] = int(m.group(0))
    return out

def pick_pz_index(ch_names, pz_name="Pz"):
    lower = [c.lower() for c in ch_names]
    if pz_name.lower() in lower:
        return int(lower.index(pz_name.lower()))
    for alt in ["cpz", "poz"]:
        if alt in lower:
            print(f"  WARNING: {pz_name} not found. Falling back to {alt.upper()} for this subject.")
            return int(lower.index(alt))
    raise RuntimeError(f"Neither {pz_name} nor CPz/POz found in channels: {ch_names}")


# LOAD PARTICIPANTS META
participants = pd.read_csv(PARTICIPANTS_TSV, sep="\t", dtype={"participant_id": str})
participants["participant_id"] = participants["participant_id"].astype(str)

# MAIN EXTRACTION
code2bin = parse_bdf_code2bin(BDF_PATH)
print("Parsed BDF code->bin mapping for", len(code2bin), "codes")

set_paths = sorted(glob.glob(os.path.join(AUTO_DIR, "*task-visualoddball*_autoAR.set")))
if not set_paths:
    raise RuntimeError(f"No *task-visualoddball*_autoAR.set files found in {AUTO_DIR}")

all_rows = []
summary_rows = []

for set_path in set_paths:
    fname = os.path.basename(set_path)
    m = re.match(r"(sub-[^_]+)_task-([^-_]+)_autoAR\.set", fname)
    if not m:
        print(f"Skipping {fname}: cannot parse subject/task from name")
        continue

    sub_id, task = m.group(1), m.group(2)
    print(f"\nSubject {sub_id}, task {task}")

    # Load JSON codes
    try:
        events_json = find_events_json_for_subject(sub_id, task=task)
        target_codes, standard_codes = get_target_and_standard_codes(events_json)
    except Exception as e:
        print("  ERROR reading events JSON:", e)
        continue

    print(f"  target codes from JSON: {target_codes}")
    print(f"  standard codes from JSON: {standard_codes}")

    # Load epochs
    epochs = mne.io.read_epochs_eeglab(set_path, verbose="ERROR")

    # Baseline-correct only if not already baseline-corrected
    if epochs.baseline is None:
        epochs = epochs.copy().apply_baseline(BASELINE)
        print(f"  applied baseline correction: {BASELINE}")
    else:
        print(f"  epochs already have baseline: {epochs.baseline} (no re-baseline applied)")

    data = epochs.get_data()
    times = epochs.times
    ch_names = epochs.ch_names
    n_epochs, n_ch, n_times = data.shape
    print(f"  epochs shape: {data.shape}")

    # Decode trial codes
    trial_codes = decode_trial_codes_from_mne(epochs)
    decoded = trial_codes[~np.isnan(trial_codes)].astype(int)
    unique_codes = sorted(set(decoded.tolist()))
    print(f"  unique numeric trial codes (decoded): {unique_codes}")

    # Quick overlap check with BDF mapping (debug)
    overlap_with_bdf = sorted(set(unique_codes).intersection(set(code2bin.keys())))
    print(f"  overlap(decoded codes, BDF codes) count: {len(overlap_with_bdf)}")
    if len(overlap_with_bdf) == 0:
        print("  WARNING: No overlap with BDF code map. Bin numbers may be -1 for all trials.")

    # Pick Pz
    try:
        pz_idx = pick_pz_index(ch_names, P3B_CH)
    except Exception as e:
        print("  ERROR:", e)
        continue
    print(f"  Using channel for P3b extraction: {ch_names[pz_idx]}")

    # Reject mask
    rejglobal = load_rejglobal_from_set(set_path)
    if rejglobal is not None and rejglobal.size == n_epochs:
        good_mask = ~rejglobal
    else:
        if rejglobal is not None and rejglobal.size != n_epochs:
            print(f"  Warning: rejglobal length {rejglobal.size} != n_epochs {n_epochs}; ignoring rejglobal")
        good_mask = np.ones(n_epochs, dtype=bool)

    # Time window indices
    tmin, tmax = P3B_WINDOW
    win_mask = (times >= tmin) & (times <= tmax)
    if not win_mask.any():
        raise RuntimeError(
            f"P3b window {P3B_WINDOW} outside epoch time range {times[0]:.3f} to {times[-1]:.3f} s"
        )

    # Extract features for good trials that are either target OR standard
    n_good = int(good_mask.sum())
    n_target = 0
    n_standard = 0

    for ei in range(n_epochs):
        if not good_mask[ei]:
            continue

        code = trial_codes[ei]
        if np.isnan(code):
            continue
        code = int(code)

        is_target = code in target_codes
        is_standard = code in standard_codes
        if not (is_target or is_standard):
            continue

        if is_target:
            n_target += 1
            cond = "target"
        else:
            n_standard += 1
            cond = "standard"

        # Single-channel time series at Pz (or fallback channel)
        pz_ts = data[ei, pz_idx, :]
        window_ts = pz_ts[win_mask]

        mean_amp_v = float(window_ts.mean())
        mean_amp_uv = mean_amp_v * 1e6  # convert to microvolts to match the PDF

        peak_idx = int(np.argmax(window_ts))
        peak_amp_v = float(window_ts[peak_idx])
        peak_amp_uv = peak_amp_v * 1e6
        peak_latency_ms = float(times[win_mask][peak_idx] * 1000.0)

        bin_num = int(code2bin.get(code, -1))

        all_rows.append({
            "subject": sub_id,
            "task": task,
            "epoch_index": ei,             # 0-based index into epochs
            "code": code,
            "bin": bin_num,
            "condition": cond,             # "target" or "standard"
            "is_target": bool(is_target),

            # Paper-aligned P3b measure:
            "pz_mean_300_600_uv": mean_amp_uv,

            # Extra useful diagnostics:
            "pz_mean_300_600_v": mean_amp_v,
            "pz_peak_300_600_uv": peak_amp_uv,
            "pz_peak_300_600_v": peak_amp_v,
            "pz_peak_latency_ms": peak_latency_ms,
            "pz_channel_used": ch_names[pz_idx],
        })

    print(f"  good epochs: {n_good} / {n_epochs}")
    print(f"  extracted target trials: {n_target}")
    print(f"  extracted standard trials: {n_standard}")

    summary_rows.append({
        "subject": sub_id,
        "task": task,
        "n_epochs_total": n_epochs,
        "n_good_epochs": n_good,
        "n_targets_extracted": n_target,
        "n_standards_extracted": n_standard,
        "pz_channel_used": ch_names[pz_idx],
    })

# Build trial-level dataframe
features = pd.DataFrame(all_rows)

# Merge participant metadata
features = features.merge(
    participants,
    left_on="subject",
    right_on="participant_id",
    how="left",
    validate="many_to_one"
).drop(columns=["participant_id"])

# Save trial-level
features.to_csv(OUT_TRIAL_CSV, index=False)
print(f"\nWrote {len(features)} trial rows to: {OUT_TRIAL_CSV}")

# Build subject-level summary features (target mean, standard mean, diff)
if len(features) > 0:
    subj = (
        features
        .groupby(["subject"], as_index=False)
        .agg(
            n_trials=("pz_mean_300_600_uv", "count"),
            n_targets=("is_target", "sum"),
            p3b_target_mean_uv=("pz_mean_300_600_uv", lambda x: float(np.mean(x[features.loc[x.index, "is_target"]])) if np.any(features.loc[x.index, "is_target"]) else np.nan),
            p3b_standard_mean_uv=("pz_mean_300_600_uv", lambda x: float(np.mean(x[~features.loc[x.index, "is_target"]])) if np.any(~features.loc[x.index, "is_target"]) else np.nan),
        )
    )
    subj["p3b_target_minus_standard_uv"] = subj["p3b_target_mean_uv"] - subj["p3b_standard_mean_uv"]

    # Merge meta (one row per subject from participants)
    subj = subj.merge(
        participants,
        left_on="subject",
        right_on="participant_id",
        how="left",
        validate="one_to_one"
    ).drop(columns=["participant_id"])

    subj.to_csv(OUT_SUBJ_CSV, index=False)
    print(f"Wrote subject-level features to: {OUT_SUBJ_CSV}")

diag = pd.DataFrame(summary_rows)
print("\nExtraction diagnostics (first 10 rows):")
print(diag.head(10))


Parsed BDF code->bin mapping for 27 codes

Subject sub-001, task visualoddball
  target codes from JSON: [11, 22, 33, 44, 55]
  standard codes from JSON: [12, 13, 14, 15, 21, 23, 24, 25, 31, 32, 34, 35, 41, 42, 43, 45, 51, 52, 53, 54]
Applying baseline correction (mode: mean)
  applied baseline correction: (-0.2, 0.0)
  epochs shape: (80, 35, 501)
  unique numeric trial codes (decoded): [11, 12, 13, 14, 15, 21, 22, 23, 24, 25]
  overlap(decoded codes, BDF codes) count: 10
  Using channel for P3b extraction: Pz
  good epochs: 77 / 80
  extracted target trials: 16
  extracted standard trials: 61

Subject sub-002, task visualoddball
  target codes from JSON: [11, 22, 33, 44, 55]
  standard codes from JSON: [12, 13, 14, 15, 21, 23, 24, 25, 31, 32, 34, 35, 41, 42, 43, 45, 51, 52, 53, 54]
Applying baseline correction (mode: mean)
  applied baseline correction: (-0.2, 0.0)
  epochs shape: (80, 35, 501)
  unique numeric trial codes (decoded): [11, 12, 13, 14, 15, 21, 22, 23, 24, 25]
  overlap(